# Calibration Junyi A2/A3 trên CPU miễn phí

Notebook nối các dự đoán GPU đã có với nhãn calibration, audit đủ1.621.484target/15.847người học rồi fit temperature V1. Không training, không GPU, không Modal compute, không Holdout. Source/split/checkpoint giữ pin từ lượt17/11.

Chọn runtime CPU. Đăng nhập Drive Sang khi được hỏi. Output nghiên cứu được upload vào thư mục KLTN đã chốt và kiểm SHA. State chứa URI upload được giữ riêng trong `MyDrive/KLTN_AKT_PRIVATE/calibration-free-cpu-20261117-01`, không chia sẻ thư mục này. Binary chỉ tải về scratch Colab, không tải về máy cá nhân.

Sau khi raw đã audit và upload đủ, notebook tiếp tục được từ raw trên Drive khi runtime mất. Nếu dừng trước checkpoint raw, lần tiếp theo có thể phải nối raw lại từ dự đoán đã lưu, không chạy lại GPU. Fit bị ngắt chạy lại V1 từ raw. Chưa có số đo thời gian đầy đủ trên Colab.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')


In [ ]:
import json,sys,base64,subprocess,venv,os
from pathlib import Path
TOOLS=Path("/content/kltn-free-cpu-tools"); TOOLS.mkdir(exist_ok=True)
FILES={'free_cpu_calibration.py': '"""Launcher Colab CPU, source KT và thuật toán V1 giữ nguyên."""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport stat\nimport zipfile\n\nACCOUNT = \'honguyenminhsang2005@gmail.com\'\nPARENT = \'1vaojIEDvwtE-vZgjlybwzrsjrGNE_8O7\'\nVOLUME = Path(\'/__modal/volumes/vo-dv1975utRQbmagCjA0brSe\')\nRUN = \'junyi-calibration-real-20261117-01\'\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef need(value, reason):\n    if not value:\n        raise ValueError(reason)\n\n\ndef physical_inside(path, root):\n    path, root = Path(path).absolute(), Path(root).resolve(strict=True)\n    need(path.is_relative_to(root) and path.resolve() == path, \'Noncanonical scratch path\')\n    for component in (path, *path.parents):\n        if component == root:\n            break\n        need(not component.is_symlink(), \'Scratch alias rejected\')\n    return path\n\n\ndef publication_source(publication):\n    match = re.fullmatch(r\'(a[23])-(raw|fit)-freecpu-\\1-\\d{3}\', publication[\'tranche_id\'])\n    need(publication[\'run_id\'] == RUN and match, \'Publication identity rejected\')\n    files = set(publication[\'files\'])\n    raw = {\'config.json\', \'RAW_EXPORT.zip\', \'COMPLETE.json\'}\n    fit = {\'config.json\', \'fit/temperature.json\', \'fit/manifest.json\', \'resource.json\', \'COMPLETE.json\'}\n    expected = raw if match[2] == \'raw\' else fit\n    need(len(files) == len(publication[\'files\']) and files == expected, \'Publication file allowlist rejected\')\n    source = physical_inside(VOLUME / RUN / publication[\'tranche_id\'] / \'payloads\', VOLUME)\n    for name in files:\n        physical_inside(source / name, VOLUME)\n    return source\n\n\ndef download(session, pin, destination):\n    """Range resume chỉ trên scratch Colab, kiểm SHA trước công bố."""\n    destination = Path(destination)\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        need(destination.stat().st_size == pin[\'bytes\'] and sha(destination) == pin[\'sha256\'], \'Existing download changed\')\n        return destination\n    partial = destination.with_name(destination.name + \'.part\')\n    offset = partial.stat().st_size if partial.exists() else 0\n    need(0 <= offset <= pin[\'bytes\'], \'Partial exceeds source size\')\n    if offset < pin[\'bytes\']:\n        headers = {\'Range\': f\'bytes={offset}-\'} if offset else {}\n        with session.get(\'https://www.googleapis.com/drive/v3/files/\' + pin[\'file_id\'],\n                         params={\'alt\': \'media\'}, headers=headers, stream=True, timeout=90) as response:\n            need(response.status_code == (206 if offset else 200), \'Drive download status mismatch\')\n            if offset:\n                need(response.headers.get(\'Content-Range\') == f"bytes {offset}-{pin[\'bytes\'] - 1}/{pin[\'bytes\']}", \'Drive Range mismatch\')\n            with partial.open(\'ab\') as output:\n                for block in response.iter_content(1024**2):\n                    if not block:\n                        continue\n                    offset += len(block)\n                    need(offset <= pin[\'bytes\'], \'Download exceeds pin\')\n                    output.write(block)\n    need(partial.stat().st_size == pin[\'bytes\'] and sha(partial) == pin[\'sha256\'], \'Download size/SHA mismatch\')\n    os.replace(partial, destination)\n    return destination\n\n\ndef extract_numeric(archive_path, pin, root, variant):\n    need(sha(archive_path) == pin[\'sha256\'], \'Numeric archive SHA changed\')\n    Path(root).mkdir(parents=True, exist_ok=True)\n    prefix = f\'{RUN}/{variant.lower()}-numeric/payloads/\'\n    with zipfile.ZipFile(archive_path) as archive:\n        members = archive.infolist()\n        need(len(members) == pin[\'source_members\'] and len({m.filename for m in members}) == len(members), \'Numeric member count mismatch\')\n        for info in members:\n            path = PurePosixPath(info.filename)\n            need(info.filename.startswith(prefix) and str(path) == info.filename\n                 and not path.is_absolute() and \'..\' not in path.parts and \'\\\\\' not in info.filename\n                 and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16)\n                 and 0 < info.file_size <= 2 * 1024**2, \'Unsafe numeric member\')\n            destination = physical_inside(Path(root) / info.filename, root)\n            destination.parent.mkdir(parents=True, exist_ok=True)\n            data = archive.read(info)\n            if destination.exists():\n                need(destination.read_bytes() == data, \'Restaged numeric changed\')\n            else:\n                destination.write_bytes(data)\n\n\ndef publish(session, publication, state_root):\n    """Reuse uploader, URI tiếp tục chỉ giữ trong state Drive riêng."""\n    import drive_resumable_upload as core\n    source = publication_source(publication)\n    rows = [dict(path=name, **core.hashes(source / name)) for name in publication[\'files\']]\n    total = sum(row[\'bytes\'] for row in rows)\n    need(0 < total <= publication[\'max_total_bytes\'] <= 2 * 1024**3, \'Publication exceeds byte contract\')\n    plan = dict(source=str(source), files=rows, bytes=total,\n                folder_name=publication[\'tranche_id\'], parent_id=PARENT, account=ACCOUNT)\n    private = Path(state_root) / \'.sessions\' / publication[\'tranche_id\']\n    evidence = Path(state_root) / \'publications\' / publication[\'tranche_id\']\n    private.mkdir(parents=True, exist_ok=True)\n    evidence.mkdir(parents=True, exist_ok=True)\n    state_path = private / \'state.json\'\n    with core.single_writer(private / \'upload.lock\'):\n        if state_path.exists():\n            previous = core.read_json(state_path)\n            strip = lambda p: dict(p, files=[{k: v for k, v in r.items() if k != \'mtime_ns\'} for r in p[\'files\']])\n            need(strip(previous[\'plan\']) == strip(plan), \'Persisted upload payload changed\')\n            previous[\'plan\'] = plan\n            for entry in previous[\'files\'].values():\n                entry.pop(\'local\', None)\n            core.atomic_json(state_path, previous)\n        uploader = core.Uploader(session, plan, state_path, evidence, chunk_bytes=8 * 1024**2, retries=2)\n        uploader.run()\n    files = []\n    for row in rows:\n        meta = uploader.metadata(uploader.state[\'files\'][row[\'path\']][\'remote_id\'])\n        parent = uploader.state[\'folders\'][\'.\'] if \'/\' not in row[\'path\'] else uploader.state[\'folders\'][str(Path(row[\'path\']).parent)]\n        need(uploader.matches(meta, row, parent) and meta.get(\'sha256Checksum\') == row[\'sha256\'], \'Drive strict SHA readback failed\')\n        files.append(dict(path=row[\'path\'], file_id=meta[\'id\'], folder_id=parent,\n                          bytes=row[\'bytes\'], sha256=row[\'sha256\']))\n    return dict(status=\'DRIVE_SHA256_VERIFIED\', tranche_id=publication[\'tranche_id\'], files=files)\n\n\ndef verify_publication(session, receipt):\n    """Đọc lại metadata Drive trước khi dùng checkpoint vận hành đã hoàn tất."""\n    need(receipt[\'status\'] == \'DRIVE_SHA256_VERIFIED\' and receipt[\'files\'], \'Publication was not verified\')\n    for pin in receipt[\'files\']:\n        response = session.get(\'https://www.googleapis.com/drive/v3/files/\' + pin[\'file_id\'],\n            params={\'fields\': \'id,name,size,sha256Checksum,parents,trashed\'}, timeout=60)\n        need(response.status_code == 200, \'Published artifact unavailable\')\n        meta = response.json()\n        need(not meta.get(\'trashed\') and meta.get(\'sha256Checksum\') == pin[\'sha256\']\n             and int(meta.get(\'size\', -1)) == pin[\'bytes\']\n             and pin[\'folder_id\'] in meta.get(\'parents\', [])\n             and meta.get(\'name\') == Path(pin[\'path\']).name, \'Published artifact changed\')\n\n\ndef publication_pins(publication):\n    source = publication_source(publication)\n    return {name: {\'bytes\': (source / name).stat().st_size, \'sha256\': sha(source / name)}\n            for name in publication[\'files\']}\n\n\ndef pending_exists(publication, expected):\n    source = publication_source(publication)\n    if not all((source / name).is_file() for name in publication[\'files\']):\n        return False\n    need(publication_pins(publication) == expected, \'Pending artifact bytes changed\')\n    return True\n\n\ndef restore_raw(session, publication):\n    need(re.fullmatch(r\'a[23]-raw-freecpu-a[23]-\\d{3}\', publication[\'tranche_id\']), \'Raw tranche rejected\')\n    payloads = physical_inside(VOLUME / RUN / publication[\'tranche_id\'] / \'payloads\', VOLUME)\n    raw = payloads / \'raw\'\n    pin = next(row for row in publication[\'files\'] if row[\'path\'] == \'RAW_EXPORT.zip\')\n    archive = download(session, pin, payloads / \'RAW_EXPORT.zip\')\n    with zipfile.ZipFile(archive) as zipped:\n        infos = zipped.infolist()\n        need(len(infos) <= 16000 and len({row.filename for row in infos}) == len(infos)\n             and sum(row.file_size for row in infos) <= 2 * 1024**3\n             and all(0 < row.file_size <= 64 * 1024**2 and not stat.S_ISLNK(row.external_attr >> 16)\n                     for row in infos), \'Raw archive size/type coverage rejected\')\n        need(zipped.getinfo(\'INVENTORY.json\').file_size <= 8 * 1024**2, \'Raw inventory exceeds budget\')\n        inventory = json.loads(zipped.read(\'INVENTORY.json\'))\n        need(set(zipped.namelist()) == {\'INVENTORY.json\'} | {\'raw/\' + name for name in inventory}, \'Raw archive coverage changed\')\n        for name, pin in inventory.items():\n            need(Path(name).name == name and 0 < pin[\'bytes\'] <= 64 * 1024**2, \'Bounded raw basename required\')\n            data = zipped.read(\'raw/\' + name)\n            need(len(data) == pin[\'bytes\'] and hashlib.sha256(data).hexdigest() == pin[\'sha256\'], \'Raw archive member changed\')\n            raw.mkdir(exist_ok=True)\n            target = physical_inside(raw / name, VOLUME)\n            if target.exists():\n                need(sha(target) == pin[\'sha256\'], \'Existing raw shard changed\')\n            else:\n                target.write_bytes(data)\n    return raw\n\n\ndef run_variant(request, gpu_result, cache, session, state_root):\n    """Nối/audit/fit V1, tiếp tục từ raw đã kiểm và lưu trên Drive."""\n    from dataclasses import asdict\n    from knowledge_tracing.akt.config import DataPaths, PROJECT_ROOT\n    from knowledge_tracing.akt.research_junyi_inference import JunyiCPUInferenceBudget, inspect_cpu_refit_export, _identity\n    from knowledge_tracing.akt.research_junyi_calibration import CalibrationFitArtifactRoot, inference_source_identity\n    from knowledge_tracing.akt.research_junyi_numeric_predictions import NumericBinding\n    from knowledge_tracing.akt.cloud_calibration_numeric_join import join_numeric_calibration_snapshot\n    from knowledge_tracing.akt.cloud_calibration_fit import fit_volume_calibration\n    from knowledge_tracing.akt.research_junyi_graph_cache import _digest, FORMAT\n    from knowledge_tracing.response_calibration import PredictionBinding\n    from calibration_audit_transaction import transactional_expected_insert\n    import drive_resumable_upload as core\n    variant = request[\'variant\']\n    need(variant in (\'A2\', \'A3\') and request[\'mode\'] == \'full\' and request[\'full_cohort_approved\'], \'Approved full calibration A2/A3 required\')\n    need(inference_source_identity() == request[\'source_identity\'], \'Frozen inference source changed\')\n    need(hashlib.sha256(json.dumps(cache).encode()).hexdigest() == request[\'cache_receipt\'][\'sha256\'], \'Cache receipt changed\')\n    for entry in [*request[\'exports\'].values(), *request[\'data_paths\'].values(), request[\'population\']]:\n        need(sha(entry[\'path\']) == entry[\'sha256\'], \'Input pin changed\')\n    index = request[\'exports\'][variant]\n    assets = inspect_cpu_refit_export(Path(index[\'path\']).parent, expected_index_sha256=index[\'sha256\'],\n        output=PROJECT_ROOT / \'experiments/akt/results\' / (\'free-cpu-\' + variant.lower()),\n        run_id=\'free-cpu-\' + variant.lower(), budget=JunyiCPUInferenceBudget(**request[\'cpu_budget\']))\n    paths = DataPaths(**{key: Path(pin[\'path\']) for key, pin in request[\'data_paths\'].items()})\n    population = json.loads(Path(request[\'population\'][\'path\']).read_text())\n    need(population[\'source_pins\'] == request[\'source_pins\'] and population == cache[\'population\'], \'Population/source mismatch\')\n    receipt_path = Path(gpu_result[\'receipt_path\'])\n    need(sha(receipt_path) == gpu_result[\'receipt_sha256\'], \'GPU receipt changed\')\n    numeric = json.loads(receipt_path.read_text())\n    binding = PredictionBinding(assets[\'index\'][\'model\'], assets[\'index\'][\'checkpoint_sha256\'],\n        assets[\'encoding\'].identity, assets[\'calibration\'].roles.identity,\n        _identity(dict(domain=\'junyi_exercises\', values=assets[\'encoding\'].native.vocabulary.values)),\n        _identity(dict(domain=\'junyi_logged_response\', reference_bank=\'not_applicable\')),\n        request[\'source_identity\'], cache[\'source_data_sha256\'])\n    graph = cache[\'graphs\'][variant]\n    graph_identity = _digest((FORMAT, graph[\'sha256\'], json.dumps(graph[\'binding\'], sort_keys=True)))\n    root = CalibrationFitArtifactRoot(VOLUME)\n    state_file = Path(state_root) / (variant + \'_PROGRESS.json\')\n    fingerprint = _identity(dict(request=request, cache=cache, binding=asdict(binding), gpu_result=gpu_result))\n    state = core.read_json(state_file) if state_file.exists() else dict(variant=variant, fingerprint=fingerprint,\n        gpu_receipt_sha256=gpu_result[\'receipt_sha256\'], attempts=0)\n    need(state[\'gpu_receipt_sha256\'] == gpu_result[\'receipt_sha256\']\n         and state[\'fingerprint\'] == fingerprint, \'Progress belongs to different inputs\')\n    if state.get(\'fit_verified\'):\n        verify_publication(session, state[\'raw_publication\'])\n        verify_publication(session, state[\'fit_publication\'])\n        return state\n    if state.get(\'pending_fit\') and pending_exists(state[\'pending_fit\'], state[\'pending_fit_files\']):\n        verify_publication(session, state[\'raw_publication\'])\n        state.update(fit_verified=True, fit_publication=publish(session, state[\'pending_fit\'], state_root))\n        state.pop(\'pending_fit\')\n        state.pop(\'pending_fit_files\')\n        core.atomic_json(state_file, state)\n        return state\n    if state.get(\'pending_raw\'):\n        publication = state[\'pending_raw\']\n        if pending_exists(publication, state[\'pending_raw_files\']):\n            state.update(raw_verified=True, raw_publication=publish(session, publication, state_root))\n            state.pop(\'pending_raw\')\n            state.pop(\'pending_raw_files\')\n            core.atomic_json(state_file, state)\n    state[\'attempts\'] += 1\n    attempt = f"freecpu-{variant.lower()}-{state[\'attempts\']:03d}"\n    core.atomic_json(state_file, state)\n    fit_tranche = variant.lower() + \'-fit-\' + attempt\n    if state.get(\'raw_verified\'):\n        verify_publication(session, state[\'raw_publication\'])\n        raw = restore_raw(session, state[\'raw_publication\'])\n        manifest = json.loads((raw / \'manifest.json\').read_text())\n        fitted = fit_volume_calibration(raw, paths, assets[\'calibration\'], binding=binding,\n            population=population, expected_population_identity=population[\'identity\'],\n            expected_manifest_sha256=sha(raw / \'manifest.json\'),\n            shard_sha256={row[\'path\']: sha(raw / row[\'path\']) for row in manifest[\'shards\']},\n            source_pins=request[\'source_pins\'], max_targets=population[\'targets\'],\n            max_learner_events=request[\'cpu_budget\'][\'max_learner_events\'],\n            artifact_root=root, run_id=RUN, tranche_id=fit_tranche)\n        fit_request = fitted[\'publish_request\']\n    else:\n        def raw_ready(publication):\n            state[\'pending_raw\'] = publication\n            state[\'pending_raw_files\'] = publication_pins(publication)\n            core.atomic_json(state_file, state)\n            receipt = publish(session, publication, state_root)\n            state.update(raw_verified=True, raw_publication=receipt)\n            state.pop(\'pending_raw\', None)\n            state.pop(\'pending_raw_files\', None)\n            core.atomic_json(state_file, state)\n        win = cache[\'windows\']\n        with transactional_expected_insert():\n            result = join_numeric_calibration_snapshot(paths, assets[\'calibration\'], assets[\'encoding\'],\n                binding=binding, numeric_binding=NumericBinding(**numeric[\'binding\']),\n                gpu_receipt=receipt_path, expected_gpu_receipt_sha256=gpu_result[\'receipt_sha256\'],\n                window_manifest=Path(win[\'path\']) / \'manifest.json\', window_admission=Path(win[\'admission_path\']),\n                expected_window_admission_sha256=win[\'admission_sha256\'], population=population,\n                expected_population_identity=population[\'identity\'], source_pins=request[\'source_pins\'],\n                max_targets=population[\'targets\'], max_learner_events=request[\'cpu_budget\'][\'max_learner_events\'],\n                artifact_root=root, run_id=RUN, raw_tranche_id=variant.lower() + \'-raw-\' + attempt,\n                fit_tranche_id=fit_tranche, expected_graph_cache_identity=graph_identity,\n                on_raw_ready=raw_ready)\n        fit_request = result[\'publish_requests\'][1]\n    state.update(pending_fit=fit_request, pending_fit_files=publication_pins(fit_request))\n    core.atomic_json(state_file, state)\n    state.update(fit_verified=True, fit_publication=publish(session, fit_request, state_root))\n    state.pop(\'pending_fit\')\n    state.pop(\'pending_fit_files\')\n    core.atomic_json(state_file, state)\n    return state\n', 'drive_resumable_upload.py': '"""Upload archive AKT lên Drive, lưu phiên riêng để tiếp tục sau mất mạng."""\nfrom __future__ import annotations\n\nimport argparse\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport mimetypes\nimport os\nfrom pathlib import Path\nimport re\nimport time\nfrom urllib.parse import urlparse\n\nimport requests\nfrom google.auth.exceptions import TransportError\n\nROOT = Path(__file__).resolve().parents[2]\nSOURCE = ROOT / \'experiments/akt/results/drive-offload-history-20261116-01\'\nPRIVATE = ROOT / \'artifacts/akt/private/drive-upload\'\nEVIDENCE = ROOT / \'research_tracking/runs/drive-history-upload-20261116-01\'\nAPI = \'https://www.googleapis.com/drive/v3\'\nUPLOAD = \'https://www.googleapis.com/upload/drive/v3/files\'\nFOLDER = \'application/vnd.google-apps.folder\'\nFIELDS = \'id,name,size,md5Checksum,sha256Checksum,parents,mimeType,trashed\'\nSCOPE = \'https://www.googleapis.com/auth/drive\'\n\n\ndef now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef read_json(path):\n    return json.loads(path.read_text(encoding=\'utf-8-sig\'))\n\n\ndef atomic_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    with temporary.open(\'w\', encoding=\'utf-8\') as stream:\n        json.dump(value, stream, ensure_ascii=False, indent=2)\n        stream.write(\'\\n\')\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temporary, path)\n\n\ndef fingerprint(path):\n    stat = path.stat()\n    return dict(bytes=stat.st_size, mtime_ns=stat.st_mtime_ns)\n\n\ndef hashes(path):\n    before = fingerprint(path)\n    sha = hashlib.sha256()\n    md5 = hashlib.md5(usedforsecurity=False)\n    with path.open(\'rb\') as stream:\n        while block := stream.read(1024 * 1024):\n            sha.update(block)\n            md5.update(block)\n    if before != fingerprint(path):\n        raise RuntimeError(\'File nguồn thay đổi trong lúc hash.\')\n    return dict(**before, sha256=sha.hexdigest(), md5=md5.hexdigest())\n\n\ndef build_plan(source, parent_id, account):\n    source = source.resolve()\n    archive_manifest = read_json(source / \'MANIFEST.json\')\n    expected = {row[\'upload_path\']: row for row in archive_manifest[\'files\']}\n    files = []\n    for path in sorted(source.rglob(\'*\')):\n        if path.is_symlink() or path.is_junction():\n            raise RuntimeError(\'Thư mục upload có symlink/junction, cần rà riêng.\')\n        if not path.is_file():\n            continue\n        relative = path.relative_to(source).as_posix()\n        stat = fingerprint(path)\n        if stat[\'bytes\'] == 0:\n            raise RuntimeError(f\'File rỗng cần xử lý riêng trước upload: {relative}\')\n        original = expected.get(relative)\n        if original:\n            if stat[\'bytes\'] != original[\'bytes\']:\n                raise RuntimeError(f\'Kích thước archive đã đổi: {relative}\')\n            digest = original[\'sha256\']\n        else:\n            digest = hashes(path)[\'sha256\']\n        files.append(dict(path=relative, **stat, sha256=digest))\n    if set(expected) - {row[\'path\'] for row in files}:\n        raise RuntimeError(\'Thiếu archive trong manifest.\')\n    return dict(source=str(source), parent_id=parent_id, account=account,\n                folder_name=source.name, files=files,\n                bytes=sum(row[\'bytes\'] for row in files),\n                note_vi=\'SHA archive lấy từ inventory lịch sử, sẽ hash thực tế từng file trước upload.\')\n\n\ndef checked_offset(response, total):\n    if response.status_code in (200, 201):\n        return total\n    if response.status_code != 308:\n        raise RuntimeError(f\'Phản hồi phiên upload không hợp lệ, HTTP {response.status_code}.\')\n    received = response.headers.get(\'Range\')\n    if not received:\n        return 0\n    match = re.fullmatch(r\'bytes=0-(\\d+)\', received)\n    if not match or not 0 < int(match[1]) + 1 <= total:\n        raise RuntimeError(\'Drive trả vị trí byte không hợp lệ.\')\n    return int(match[1]) + 1\n\n\ndef safe_session_url(url):\n    parsed = urlparse(url)\n    if (parsed.scheme != \'https\' or parsed.hostname != \'www.googleapis.com\'\n            or parsed.username or parsed.password or parsed.port not in (None, 443)\n            or not parsed.path.startswith(\'/upload/drive/v3/files\')):\n        raise RuntimeError(\'Địa chỉ phiên upload không thuộc endpoint Drive được phép.\')\n    return url\n\n\n@contextmanager\ndef single_writer(path):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'a+b\') as stream:\n        stream.seek(0, os.SEEK_END)\n        if stream.tell() == 0:\n            stream.write(b\'0\')\n            stream.flush()\n        stream.seek(0)\n        try:\n            if os.name == \'nt\':\n                import msvcrt\n                msvcrt.locking(stream.fileno(), msvcrt.LK_NBLCK, 1)\n            else:\n                import fcntl\n                fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except OSError:\n            raise RuntimeError(\'Một uploader khác đang dùng trạng thái này.\') from None\n        try:\n            yield\n        finally:\n            stream.seek(0)\n            if os.name == \'nt\':\n                msvcrt.locking(stream.fileno(), msvcrt.LK_UNLCK, 1)\n            else:\n                fcntl.flock(stream.fileno(), fcntl.LOCK_UN)\n\n\nclass Uploader:\n    def __init__(self, session, plan, state_path, evidence, chunk_bytes=32 * 1024**2,\n                 retries=20, sleep=time.sleep):\n        if chunk_bytes <= 0 or chunk_bytes % (256 * 1024):\n            raise ValueError(\'Chunk phải là bội dương của 256 KiB.\')\n        self.session = session\n        self.plan = plan\n        self.state_path = state_path\n        self.evidence = evidence\n        self.chunk_bytes = chunk_bytes\n        self.retries = retries\n        self.sleep = sleep\n        self.state = dict(plan=plan, files={}, folders={})\n        if state_path.exists():\n            self.state = read_json(state_path)\n            if self.state[\'plan\'] != plan:\n                raise RuntimeError(\'Nguồn/đích hoặc inventory đã đổi. Giữ state cũ để rà trước.\')\n\n    def save(self):\n        atomic_json(self.state_path, self.state)\n\n    def event(self, action, **fields):\n        self.evidence.mkdir(parents=True, exist_ok=True)\n        with (self.evidence / \'events.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n            stream.write(json.dumps(dict(at=now(), action=action, **fields), ensure_ascii=False) + \'\\n\')\n\n    def wait(self, failures):\n        if failures > self.retries:\n            raise RuntimeError(\'Hết số lần thử lại. Chạy cùng lệnh để tiếp tục từ state đã giữ.\')\n        seconds = min(60, 2 ** min(failures, 6))\n        print(f\'Đang chờ mạng/Drive, thử lại sau {seconds}s.\', flush=True)\n        self.sleep(seconds)\n\n    def api(self, method, path, allowed=(200,), **kwargs):\n        for attempt in range(self.retries + 1):\n            try:\n                response = self.session.request(method, API + path, timeout=(20, 180),\n                                                allow_redirects=False, **kwargs)\n            except (requests.RequestException, TransportError):\n                self.wait(attempt + 1)\n                continue\n            if response.status_code in allowed:\n                return response\n            if response.status_code == 429 or response.status_code >= 500:\n                self.wait(attempt + 1)\n                continue\n            raise RuntimeError(f\'Yêu cầu Drive thất bại, HTTP {response.status_code}.\')\n        raise RuntimeError(\'Drive chưa phản hồi.\')\n\n    def metadata(self, file_id):\n        response = self.api(\'GET\', f\'/files/{file_id}\', allowed=(200, 404), params={\'fields\': FIELDS})\n        return response.json() if response.status_code == 200 else None\n\n    def reserve_id(self):\n        return self.api(\'GET\', \'/files/generateIds\', params={\'count\': 1, \'space\': \'drive\'}).json()[\'ids\'][0]\n\n    def children(self, parent, name):\n        escaped = name.replace(\'\\\\\', \'\\\\\\\\\').replace("\'", "\\\\\'")\n        query = f"\'{parent}\' in parents and name = \'{escaped}\' and trashed = false"\n        rows = []\n        page = None\n        while True:\n            params = dict(q=query, fields=f\'nextPageToken,files({FIELDS})\', pageSize=100)\n            if page:\n                params[\'pageToken\'] = page\n            result = self.api(\'GET\', \'/files\', params=params).json()\n            rows.extend(result.get(\'files\', []))\n            page = result.get(\'nextPageToken\')\n            if not page:\n                return rows\n\n    def folder(self, relative, parent, name):\n        folders = self.state[\'folders\']\n        if relative not in folders:\n            existing = [f for f in self.children(parent, name) if f[\'mimeType\'] == FOLDER]\n            if len(existing) > 1:\n                raise RuntimeError(f\'Có nhiều thư mục Drive cùng tên {name}, cần chỉ rõ đích.\')\n            folders[relative] = existing[0][\'id\'] if existing else self.reserve_id()\n            self.save()\n        file_id = folders[relative]\n        meta = self.metadata(file_id)\n        if meta is None:\n            self.api(\'POST\', \'/files\', allowed=(200, 201, 409),\n                     json=dict(id=file_id, name=name, mimeType=FOLDER, parents=[parent]))\n            meta = self.metadata(file_id)\n        if (not meta or meta.get(\'trashed\') or meta[\'mimeType\'] != FOLDER\n                or meta.get(\'name\') != name or parent not in meta.get(\'parents\', [])):\n            raise RuntimeError(\'Thư mục Drive không khớp đích đã chốt.\')\n        return file_id\n\n    def matches(self, meta, row, parent):\n        return (meta and not meta.get(\'trashed\') and int(meta.get(\'size\', -1)) == row[\'bytes\']\n                and meta.get(\'name\') == Path(row[\'path\']).name\n                and parent in meta.get(\'parents\', []) and meta.get(\'md5Checksum\') == row[\'md5\']\n                and (not meta.get(\'sha256Checksum\') or meta[\'sha256Checksum\'] == row[\'sha256\']))\n\n    def complete(self, relative, entry, meta, row, parent):\n        if not self.matches(meta, row, parent):\n            raise RuntimeError(f\'Checksum/kích thước/đích Drive chưa khớp: {relative}\')\n        entry.update(complete=True, offset=row[\'bytes\'])\n        entry.pop(\'session_url\', None)\n        entry[\'receipt\'] = dict(path=relative, file_id=meta[\'id\'], bytes=row[\'bytes\'],\n                                local_sha256=row[\'sha256\'], remote_md5=meta[\'md5Checksum\'],\n                                remote_sha256=meta.get(\'sha256Checksum\'), verified_at=now())\n        self.save()\n        atomic_json(self.evidence / \'RECEIPTS.json\',\n                    [f[\'receipt\'] for f in self.state[\'files\'].values() if f.get(\'complete\')])\n        self.event(\'FILE_VERIFIED\', **entry[\'receipt\'])\n        print(f\'Đã kiểm xong {relative}\', flush=True)\n\n    def send_file(self, row, parent):\n        relative = row[\'path\']\n        path = Path(self.plan[\'source\']) / relative\n        entry = self.state[\'files\'].setdefault(relative, {})\n        current = fingerprint(path)\n        if current != {k: row[k] for k in (\'bytes\', \'mtime_ns\')}:\n            raise RuntimeError(f\'Nguồn đã đổi: {relative}\')\n        if entry.get(\'local\') is None:\n            print(f\'Kiểm SHA nguồn {relative}\', flush=True)\n            entry[\'local\'] = hashes(path)\n            if entry[\'local\'][\'sha256\'] != row[\'sha256\']:\n                raise RuntimeError(f\'SHA nguồn không khớp inventory: {relative}\')\n            self.save()\n        if (entry[\'local\'][\'sha256\'] != row[\'sha256\']\n                or {k: entry[\'local\'][k] for k in (\'bytes\', \'mtime_ns\')} != current):\n            raise RuntimeError(f\'Hash đã lưu không khớp inventory: {relative}\')\n        row = dict(row, md5=entry[\'local\'][\'md5\'])\n        if \'remote_id\' not in entry:\n            matches = [f for f in self.children(parent, path.name) if self.matches(f, row, parent)]\n            entry[\'remote_id\'] = matches[0][\'id\'] if matches else self.reserve_id()\n            self.save()\n        meta = self.metadata(entry[\'remote_id\'])\n        if meta:\n            self.complete(relative, entry, meta, row, parent)\n            return\n        if entry.get(\'complete\'):\n            raise RuntimeError(f\'File đã nghiệm thu bị mất trên Drive: {relative}\')\n        failures = 0\n        offset = None\n        expired = 0\n        last_progress = entry.get(\'offset\') or 0\n        while True:\n            if fingerprint(path) != current:\n                raise RuntimeError(f\'Nguồn thay đổi trong lúc upload: {relative}\')\n            try:\n                if \'session_url\' not in entry:\n                    # ID được lưu trước POST, tránh tạo file trùng khi mất phản hồi cuối.\n                    response = self.session.request(\'POST\', UPLOAD, params={\'uploadType\': \'resumable\'},\n                        json=dict(id=entry[\'remote_id\'], name=path.name, parents=[parent]),\n                        headers={\'X-Upload-Content-Length\': str(row[\'bytes\']),\n                                 \'X-Upload-Content-Type\': mimetypes.guess_type(path.name)[0] or \'application/octet-stream\'},\n                        timeout=(20, 180), allow_redirects=False)\n                    if response.status_code == 409:\n                        self.complete(relative, entry, self.metadata(entry[\'remote_id\']), row, parent)\n                        return\n                    if response.status_code in (429, 500, 502, 503, 504):\n                        raise requests.ConnectionError(\'Drive tạm bận\')\n                    if response.status_code not in (200, 201):\n                        raise RuntimeError(f\'Tạo phiên thất bại, HTTP {response.status_code}.\')\n                    entry[\'session_url\'] = safe_session_url(response.headers[\'Location\'])\n                    self.save()\n                    offset = None\n                url = safe_session_url(entry[\'session_url\'])\n                if offset is None:\n                    response = self.session.request(\'PUT\', url, data=b\'\',\n                        headers={\'Content-Length\': \'0\', \'Content-Range\': f"bytes */{row[\'bytes\']}"},\n                        timeout=(20, 180), allow_redirects=False)\n                else:\n                    with path.open(\'rb\') as stream:\n                        stream.seek(offset)\n                        block = stream.read(min(self.chunk_bytes, row[\'bytes\'] - offset))\n                    end = offset + len(block) - 1\n                    response = self.session.request(\'PUT\', url, data=block,\n                        headers={\'Content-Length\': str(len(block)),\n                                 \'Content-Range\': f"bytes {offset}-{end}/{row[\'bytes\']}"},\n                        timeout=(20, 180), allow_redirects=False)\n                if response.status_code == 404:\n                    meta = self.metadata(entry[\'remote_id\'])\n                    if meta:\n                        self.complete(relative, entry, meta, row, parent)\n                        return\n                    expired += 1\n                    if expired > 2:\n                        raise RuntimeError(\'Phiên liên tục hết hạn. Giữ state để kiểm quyền/Drive.\')\n                    entry.pop(\'session_url\')\n                    self.save()\n                    self.event(\'SESSION_EXPIRED\', path=relative)\n                    continue\n                if response.status_code == 429 or response.status_code >= 500:\n                    raise requests.ConnectionError(\'Drive tạm bận\')\n                offset = checked_offset(response, row[\'bytes\'])\n                if response.status_code in (200, 201):\n                    self.complete(relative, entry, self.metadata(entry[\'remote_id\']), row, parent)\n                    return\n                if offset == row[\'bytes\']:\n                    offset = None\n                entry[\'offset\'] = offset\n                self.save()\n                if offset is not None and offset > last_progress:\n                    failures = 0\n                    last_progress = offset\n                shown_offset = row[\'bytes\'] if offset is None else offset\n                print(f"{relative}: {shown_offset / row[\'bytes\']:.1%}", flush=True)\n            except (requests.RequestException, TransportError):\n                offset = None  # Hỏi byte Drive đã nhận trước khi gửi lại sau mất mạng.\n                failures += 1\n                self.wait(failures)\n\n    def run(self, login_only=False):\n        about = self.api(\'GET\', \'/about\', params={\'fields\': \'user(emailAddress),storageQuota\'}).json()\n        account = about[\'user\'][\'emailAddress\']\n        if account.lower() != self.plan[\'account\'].lower():\n            raise RuntimeError(\'Tài khoản Drive khác tài khoản đã chốt trong kế hoạch.\')\n        quota = about.get(\'storageQuota\', {})\n        if \'limit\' in quota:\n            remaining = int(quota[\'limit\']) - int(quota.get(\'usage\', 0))\n            print(f\'Drive đang còn {remaining / 1e9:.2f} GB trống.\', flush=True)\n        parent = self.metadata(self.plan[\'parent_id\'])\n        if not parent or parent.get(\'trashed\') or parent[\'mimeType\'] != FOLDER:\n            raise RuntimeError(\'Chưa truy cập được thư mục đích Drive.\')\n        if login_only:\n            print(\'Đã kiểm đúng tài khoản và quyền đọc thư mục đích.\', flush=True)\n            return\n        root_id = self.folder(\'.\', self.plan[\'parent_id\'], self.plan[\'folder_name\'])\n        for row in self.plan[\'files\']:\n            parent = root_id\n            relative_parent = Path(row[\'path\']).parent\n            current = Path()\n            for part in relative_parent.parts:\n                current /= part\n                parent = self.folder(current.as_posix(), parent, part)\n            self.send_file(row, parent)\n        self.event(\'BATCH_VERIFIED\', files=len(self.plan[\'files\']), bytes=self.plan[\'bytes\'])\n        atomic_json(self.evidence / \'ACCEPTANCE.json\', dict(status=\'REMOTE_CHECKSUM_VERIFIED\',\n            files=len(self.plan[\'files\']), bytes=self.plan[\'bytes\'], folder_id=root_id,\n            completed_at=now(), local_files_deleted=0))\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--prepare\', action=\'store_true\', help=\'Lập inventory local, chưa kết nối Drive\')\n    parser.add_argument(\'--login-only\', action=\'store_true\', help=\'Đăng nhập và kiểm tài khoản/đích\')\n    parser.add_argument(\'--source\', type=Path, default=SOURCE)\n    parser.add_argument(\'--parent-id\', default=\'1vaojIEDvwtE-vZgjlybwzrsjrGNE_8O7\')\n    parser.add_argument(\'--account\', default=\'honguyenminhsang2005@gmail.com\')\n    parser.add_argument(\'--chunk-mib\', type=int, default=32)\n    args = parser.parse_args()\n    if not 1 <= args.chunk_mib <= 64:\n        parser.error(\'chunk-mib từ 1 đến 64.\')\n    plan_path = EVIDENCE / \'UPLOAD_PLAN.json\'\n    if args.login_only and plan_path.exists():\n        # Đăng nhập tiếp tục dùng được sau khi archive đã chuyển sang Drive.\n        plan = read_json(plan_path)\n        if (plan[\'source\'] != str(args.source.resolve()) or plan[\'parent_id\'] != args.parent_id\n                or plan[\'account\'].lower() != args.account.lower()):\n            raise RuntimeError(\'Tham số đăng nhập khác kế hoạch Drive đã lưu.\')\n    else:\n        plan = build_plan(args.source, args.parent_id, args.account)\n    if plan_path.exists() and read_json(plan_path) != plan:\n        raise RuntimeError(\'Kế hoạch đã có và khác inventory hiện tại, cần rà thay đổi trước.\')\n    if not plan_path.exists():\n        atomic_json(plan_path, plan)\n    print(f"Kế hoạch {len(plan[\'files\'])} file, {plan[\'bytes\']/1e9:.2f} GB.", flush=True)\n    if args.prepare:\n        return\n    client_path = PRIVATE / \'client_secret.json\'\n    token_path = PRIVATE / \'token.json\'\n    if not token_path.exists() and not client_path.exists():\n        raise RuntimeError(f\'Cần OAuth Desktop client tại {client_path}\')\n    from google.auth.transport.requests import AuthorizedSession\n    from google.oauth2.credentials import Credentials\n    from google_auth_oauthlib.flow import InstalledAppFlow\n    PRIVATE.mkdir(parents=True, exist_ok=True)\n    if os.name != \'nt\':\n        PRIVATE.chmod(0o700)\n    with single_writer(PRIVATE / \'upload.lock\'):\n        if token_path.exists():\n            credentials = Credentials.from_authorized_user_file(str(token_path), [SCOPE])\n        else:\n            flow = InstalledAppFlow.from_client_secrets_file(str(client_path), [SCOPE])\n            credentials = flow.run_local_server(port=0, login_hint=args.account,\n                access_type=\'offline\', prompt=\'consent\', timeout_seconds=300)\n            atomic_json(token_path, json.loads(credentials.to_json()))\n        try:\n            with AuthorizedSession(credentials) as session:\n                Uploader(session, plan, PRIVATE / \'state.json\', EVIDENCE,\n                         chunk_bytes=args.chunk_mib * 1024**2).run(login_only=args.login_only)\n        finally:\n            atomic_json(token_path, json.loads(credentials.to_json()))\n\n\nif __name__ == \'__main__\':\n    try:\n        main()\n    except KeyboardInterrupt:\n        print(\'Đã dừng. Chạy lại cùng lệnh để tiếp tục phiên đã lưu.\', flush=True)\n        raise SystemExit(130)\n    except RuntimeError as error:\n        print(str(error), flush=True)\n        raise SystemExit(1)\n    except Exception as error:\n        # Ngoại lệ HTTP/OAuth có thể chứa URL phiên hoặc token, chỉ ghi loại lỗi.\n        print(f\'Đã dừng do {type(error).__name__}; giữ trạng thái để tiếp tục/kiểm tra.\', flush=True)\n        raise SystemExit(1)\n', 'modal_calibration_input_stage.py': '"""CPU staging đầu vào calibration đã ghim từ Drive sang Modal Volume.\n\nModule chỉ định nghĩa worker. Không gọi cloud khi import hoặc chạy local.\n"""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport re\nimport stat\nimport time\nimport uuid\nimport zipfile\n\nROOT = Path(__file__).resolve().parents[2] if os.name == \'nt\' else Path(\'/opt/kltn\')\nACCOUNT = \'honguyenminhsang2005@gmail.com\'\nPARENT = \'1vaojIEDvwtE-vZgjlybwzrsjrGNE_8O7\'\nFILE_PARENT = \'1xFW0BKnfph3jZdsJABW3WWB8b5TnGCSg\'\nFILE_ID = \'1GlPhrWOLwFNeUiv1unYaEATypWmV9_Vy\'\nZIP_SHA = \'a34cee8e961baf7056e8f16159d2b9a25afcca72f3c19b3a37bd0d711aab2d6a\'\nZIP_BYTES = 134525060\nPLAN_SHA = \'0a5aea93f57206d04ee3094f1c13ae6d64aaa0864c95cf67f4e9ebcca7a7baef\'\nLOCAL_PLAN_FILE_SHA = \'a552518a231ee5d432e88080220e22ab161c196150b42addea08f0f94aed64dc\'\nAPI = \'https://www.googleapis.com/drive/v3\'\nMAX_BYTES = 160 * 1024**2\nMAX_FILE = 64 * 1024**2\nSAFE_FAILURE_MESSAGES = frozenset({\n    \'Bounded input download required\', \'Drive metadata request failed\',\n    \'Drive input account mismatch\', \'Drive input metadata pin mismatch\',\n    \'Drive input folder lineage mismatch\', \'Existing input ZIP pin mismatch\',\n    \'Partial input exceeds expected bytes\', \'Drive Range response does not match committed offset\',\n    \'Drive media download failed\', \'Drive download exceeds byte cap\',\n    \'Input download retry budget exhausted\', \'Downloaded input ZIP SHA mismatch\',\n    \'Input ZIP pin mismatch\', \'Input ZIP member limits or types rejected\',\n    \'Archive member needs string path\', \'Unsafe archive member path\',\n    \'Embedded input plan pin mismatch\', \'Input plan coverage mismatch\',\n    \'Existing staged input coverage mismatch\', \'Staged input alias rejected\',\n    \'Existing staged input pin mismatch\', \'Input member size pin mismatch\',\n    \'Extracted input member SHA mismatch\', \'Drive OAuth endpoint mismatch\',\n})\n\n\ndef record_safe_failure(error, phase, root, volume_commit):\n    """Chỉ công bố thông báo tĩnh của worker, không ghi raw HTTP/OAuth/URI."""\n    phases = {\'MOUNT\', \'AUTH\', \'DOWNLOAD\', \'EXTRACT\', \'RECEIPT\'}\n    message = str(error)\n    result = dict(status=\'CALIBRATION_INPUT_STAGE_FAILED\',\n        phase=phase if phase in phases else \'UNKNOWN\', error_type=type(error).__name__,\n        message=message if message in SAFE_FAILURE_MESSAGES else None,\n        zip_bytes=None, part_bytes=None)\n    if root is not None:\n        root = Path(root)\n        for name, key in ((\'INPUT.zip\', \'zip_bytes\'), (\'INPUT.zip.part\', \'part_bytes\')):\n            path = root/name\n            if path.is_file():\n                result[key] = path.stat().st_size\n    print(json.dumps(result, sort_keys=True), flush=True)\n    try:\n        if root is not None:\n            root.mkdir(parents=True, exist_ok=True)\n            with (root/(\'FAILED-\'+uuid.uuid4().hex+\'.json\')).open(\'x\', encoding=\'utf-8\') as stream:\n                json.dump(result, stream, sort_keys=True, indent=2)\n            volume_commit()\n    except Exception:\n        print(\'{"status":"STAGE_FAILURE_DIAGNOSTIC_PERSIST_FAILED"}\', flush=True)\n    return result\n\n\ndef sha256(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream, \'sha256\').hexdigest()\n\n\ndef _path(name):\n    if not isinstance(name, str):\n        raise ValueError(\'Archive member needs string path\')\n    path = PurePosixPath(name)\n    if (path.is_absolute() or not path.parts or \'..\' in path.parts or str(path) != name\n            or \'\\\\\' in name or \':\' in name or len(name) > 240 or any(ord(c) < 32 for c in name)\n            or any(p.lower() in {\'private\', \'.private\', \'.git\', \'holdout\'} for p in path.parts)):\n        raise ValueError(\'Unsafe archive member path\')\n    return path\n\n\ndef download_pinned_input(session, directory, *, volume_commit, file_id=FILE_ID,\n        expected_sha=ZIP_SHA, expected_bytes=ZIP_BYTES, retries=2, sleep=time.sleep):\n    """Resume HTTP Range từ .part đã commit, không giữ OAuth/URI trong state."""\n    import requests\n    if not 0 < expected_bytes <= MAX_BYTES or not callable(volume_commit):\n        raise ValueError(\'Bounded input download required\')\n    def metadata(path, params):\n        response = session.request(\'GET\', API + path, params=params, timeout=(20, 60), allow_redirects=False)\n        if response.status_code != 200:\n            raise RuntimeError(\'Drive metadata request failed\')\n        return response.json()\n    about = metadata(\'/about\', {\'fields\':\'user(emailAddress)\'})\n    if about[\'user\'][\'emailAddress\'].lower() != ACCOUNT:\n        raise ValueError(\'Drive input account mismatch\')\n    remote = metadata(\'/files/\' + file_id, {\'fields\':\'id,size,sha256Checksum,parents,trashed\'})\n    if (remote.get(\'trashed\') or remote.get(\'id\') != file_id or FILE_PARENT not in remote.get(\'parents\', [])\n            or int(remote.get(\'size\', -1)) != expected_bytes or remote.get(\'sha256Checksum\') != expected_sha):\n        raise ValueError(\'Drive input metadata pin mismatch\')\n    folder = metadata(\'/files/\' + FILE_PARENT, {\'fields\':\'id,mimeType,parents,trashed\'})\n    if (folder.get(\'trashed\') or folder.get(\'id\') != FILE_PARENT\n            or folder.get(\'mimeType\') != \'application/vnd.google-apps.folder\'\n            or PARENT not in folder.get(\'parents\', [])):\n        raise ValueError(\'Drive input folder lineage mismatch\')\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    completed = directory/\'INPUT.zip\'\n    partial = directory/\'INPUT.zip.part\'\n    if completed.exists():\n        if completed.stat().st_size != expected_bytes or sha256(completed) != expected_sha:\n            raise ValueError(\'Existing input ZIP pin mismatch\')\n        return completed\n    if partial.exists() and partial.stat().st_size > expected_bytes:\n        raise ValueError(\'Partial input exceeds expected bytes\')\n    for attempt in range(retries + 1):\n        offset = partial.stat().st_size if partial.exists() else 0\n        if offset == expected_bytes:\n            break\n        response = None\n        try:\n            response = session.request(\'GET\', API+\'/files/\'+file_id,\n                params={\'alt\':\'media\'}, headers={\'Range\':f\'bytes={offset}-\'}, stream=True,\n                timeout=(20, 90), allow_redirects=False)\n            if response.status_code in (429, 500, 502, 503, 504):\n                raise requests.ConnectionError(\'Drive temporarily unavailable\')\n            if offset:\n                match = re.fullmatch(r\'bytes (\\d+)-(\\d+)/(\\d+)\', response.headers.get(\'Content-Range\',\'\'))\n                if (response.status_code != 206 or not match or int(match[1]) != offset\n                        or int(match[2]) != expected_bytes-1 or int(match[3]) != expected_bytes):\n                    raise ValueError(\'Drive Range response does not match committed offset\')\n            elif response.status_code not in (200, 206):\n                raise RuntimeError(\'Drive media download failed\')\n            with partial.open(\'ab\') as stream:\n                for block in response.iter_content(chunk_size=8*1024**2):\n                    if not block:\n                        continue\n                    offset += len(block)\n                    if offset > expected_bytes:\n                        raise ValueError(\'Drive download exceeds byte cap\')\n                    stream.write(block)\n                    stream.flush()\n                    os.fsync(stream.fileno())\n                    volume_commit()\n            if offset != expected_bytes:\n                raise requests.ConnectionError(\'Incomplete Drive download\')\n            break\n        except requests.RequestException:\n            if attempt == retries:\n                raise RuntimeError(\'Input download retry budget exhausted\') from None\n            sleep(min(4, 2**attempt))\n        finally:\n            if response is not None:\n                response.close()\n    if sha256(partial) != expected_sha:\n        raise ValueError(\'Downloaded input ZIP SHA mismatch\')\n    os.replace(partial, completed)\n    volume_commit()\n    return completed\n\n\ndef extract_verified_input(archive_path, target, *, expected_zip_sha=ZIP_SHA, expected_plan_sha=PLAN_SHA):\n    """Đối chiếu ZIP/plan/member SHA trước công bố source directory."""\n    archive_path, target = Path(archive_path), Path(target)\n    if archive_path.stat().st_size > MAX_BYTES or sha256(archive_path) != expected_zip_sha:\n        raise ValueError(\'Input ZIP pin mismatch\')\n    with zipfile.ZipFile(archive_path) as archive:\n        members = archive.infolist()\n        names = [m.filename for m in members]\n        if (len(members) > 257 or len(set(names)) != len(names)\n                or sum(m.file_size for m in members) > MAX_BYTES\n                or any(m.file_size > MAX_FILE or stat.S_ISLNK(m.external_attr >> 16)\n                       or m.is_dir() or m.flag_bits & 1 for m in members)):\n            raise ValueError(\'Input ZIP member limits or types rejected\')\n        for name in names:\n            _path(name)\n        raw = archive.read(\'CLOUD_INPUT_PLAN.json\')\n        if len(raw) > 1024**2 or hashlib.sha256(raw).hexdigest() != expected_plan_sha:\n            raise ValueError(\'Embedded input plan pin mismatch\')\n        plan = json.loads(raw)\n        rows = plan[\'files\']\n        if (plan[\'schema\'] != \'junyi-calibration-cloud-inputs-v1\' or not 1 <= len(rows) <= 256\n                or len({r[\'path\'] for r in rows}) != len(rows)\n                or set(names) != {r[\'path\'] for r in rows} | {\'CLOUD_INPUT_PLAN.json\'}\n                or sum(r[\'bytes\'] for r in rows) != plan[\'bytes\'] or plan[\'bytes\'] > MAX_BYTES):\n            raise ValueError(\'Input plan coverage mismatch\')\n        if target.exists():\n            existing = {p.relative_to(target).as_posix() for p in target.rglob(\'*\') if p.is_file()}\n            if existing != {row[\'path\'] for row in rows}:\n                raise ValueError(\'Existing staged input coverage mismatch\')\n            for row in rows:\n                path = target/_path(row[\'path\'])\n                if any(p.is_symlink() or getattr(p, \'is_junction\', lambda: False)()\n                       for p in (path, *path.parents) if p.is_relative_to(target)):\n                    raise ValueError(\'Staged input alias rejected\')\n                if path.stat().st_size != row[\'bytes\'] or sha256(path) != row[\'sha256\']:\n                    raise ValueError(\'Existing staged input pin mismatch\')\n            return plan\n        staging = target.with_name(target.name+\'.partial-\'+uuid.uuid4().hex)\n        staging.mkdir(parents=True, exist_ok=False)\n        for row in rows:\n            relative = _path(row[\'path\'])\n            info = archive.getinfo(row[\'path\'])\n            if info.file_size != row[\'bytes\'] or not 0 <= row[\'bytes\'] <= MAX_FILE:\n                raise ValueError(\'Input member size pin mismatch\')\n            path = staging/relative\n            path.parent.mkdir(parents=True, exist_ok=True)\n            digest = hashlib.sha256()\n            with archive.open(info) as source, path.open(\'xb\') as output:\n                while block := source.read(1024**2):\n                    digest.update(block)\n                    output.write(block)\n            if digest.hexdigest() != row[\'sha256\']:\n                raise ValueError(\'Extracted input member SHA mismatch\')\n        os.replace(staging, target)\n        return plan\n\n\ndef create_app():\n    """Định nghĩa CPU worker; root quyết định app.run và hạn mức riêng."""\n    from modal_junyi_calibration import require_cpu_orchestration_authorization\n    require_cpu_orchestration_authorization()\n    import modal\n    volume = modal.Volume.from_name(\'kltn-junyi-calibration-artifacts-v1\', environment_name=\'main\', version=2)\n    image = (modal.Image.debian_slim(python_version=\'3.13\')\n        .pip_install(\'requests==2.34.2\', \'google-auth==2.62.1\')\n        .env({\'PYTHONPATH\':\'/opt/kltn/scripts/akt\'})\n        .add_local_file(str(ROOT/\'scripts/akt/modal_calibration_input_stage.py\'), \'/opt/kltn/scripts/akt/modal_calibration_input_stage.py\'))\n    app = modal.App(\'kltn-calibration-input-stage-cpu\')\n\n    @app.function(image=image, cpu=1, memory=1024, timeout=600, startup_timeout=180, serialized=True,\n        routing_region=\'ap-south\',\n        max_containers=1, retries=0, scaledown_window=2, single_use_containers=True,\n        volumes={\'/artifacts\':volume}, secrets=[modal.Secret.from_name(\n            \'kltn-drive-artifacts-sang2005-20261117\', environment_name=\'main\', required_keys=[\'KLTN_DRIVE_OAUTH_JSON\'])])\n    def stage():\n        from google.auth.transport.requests import AuthorizedSession\n        from google.oauth2.credentials import Credentials\n        phase = \'MOUNT\'\n        root = None\n        try:\n            volume.reload()\n            root = Path(\'/artifacts\').resolve(strict=True)/\'calibration-inputs-20261117-01\'\n            phase = \'AUTH\'\n            info = json.loads(os.environ[\'KLTN_DRIVE_OAUTH_JSON\'])\n            if info.get(\'token_uri\', \'https://oauth2.googleapis.com/token\') != \'https://oauth2.googleapis.com/token\':\n                raise ValueError(\'Drive OAuth endpoint mismatch\')\n            credentials = Credentials.from_authorized_user_info(info, [\'https://www.googleapis.com/auth/drive\'])\n            with AuthorizedSession(credentials) as session:\n                phase = \'DOWNLOAD\'\n                archive = download_pinned_input(session, root, volume_commit=volume.commit)\n            phase = \'EXTRACT\'\n            plan = extract_verified_input(archive, root/\'source\')\n            phase = \'RECEIPT\'\n            receipt = dict(status=\'CALIBRATION_CLOUD_INPUTS_STAGED\', source_root=str(root/\'source\'),\n                input_drive_file_id=FILE_ID, zip_sha256=ZIP_SHA, plan_sha256=PLAN_SHA,\n                local_plan_file_sha256=LOCAL_PLAN_FILE_SHA,\n                bytes=plan[\'bytes\'], files=plan[\'files\'], gpu=False)\n            receipt_path = root/\'STAGED.json\'\n            receipt_path.write_text(json.dumps(receipt, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n            volume.commit()\n            return dict(status=receipt[\'status\'], source_root=receipt[\'source_root\'],\n                files=len(plan[\'files\']), bytes=plan[\'bytes\'], receipt_path=str(receipt_path),\n                receipt_sha256=sha256(receipt_path), zip_sha256=ZIP_SHA, plan_sha256=PLAN_SHA, gpu=False)\n        except Exception as error:\n            diagnostic = record_safe_failure(error, phase, root, volume.commit)\n            raise RuntimeError(\'CPU input staging stopped: \'+diagnostic[\'phase\']+\' \'+\n                diagnostic[\'error_type\']+\' \'+(diagnostic[\'message\'] or \'unclassified error\')) from None\n    return app, stage\n', 'calibration_audit_transaction.py': '"""Wrapper CPU audit trong process riêng, giữ nguyên source KT và validation.\n\nAudit raw đã có chỉ dùng sau khi writer cũ kết thúc. Source SHA của wrapper\nđược ghi riêng vì inference_source_identity tiếp tục ghim source KT đã frozen.\n"""\nfrom contextlib import contextmanager\nimport json\nimport re\nfrom pathlib import Path\nimport time\n\nfrom knowledge_tracing.akt import prediction\nfrom knowledge_tracing.akt.provenance import sha256_file\nfrom knowledge_tracing.akt.cloud_calibration_fit import _volume_input\nfrom knowledge_tracing.akt.research_junyi_calibration import CalibrationFitArtifactRoot, _source_pins\nfrom knowledge_tracing.akt.research_junyi_data import iter_junyi_phase_learners\n\n\n@contextmanager\ndef transactional_expected_insert():\n    """Dùng helper gốc trong một transaction, restore hook khi thoát."""\n    original = prediction._expected_insert\n\n    def insert(con, factory):\n        con.execute(\'BEGIN TRANSACTION\')\n        try:\n            count = original(con, factory)\n            con.execute(\'COMMIT\')\n            return count\n        except BaseException:\n            con.execute(\'ROLLBACK\')\n            raise\n\n    prediction._expected_insert = insert\n    try:\n        yield\n    finally:\n        prediction._expected_insert = original\n\n\ndef resume_completed_raw_audit(raw, paths, view, *, binding, population, source_pins,\n        expected_manifest_sha256, writer_closed_marker, expected_writer_closed_sha256,\n        artifact_root, receipt_output):\n    """Tiếp tục audit cùng Parquet bytes sau writer-stop, không phát sinh predictions."""\n    if type(artifact_root) is not CalibrationFitArtifactRoot:\n        raise ValueError(\'typed cloud artifact root required\')\n    receipt_output = artifact_root.validate_output(receipt_output)\n    root = Path(artifact_root.root).resolve(strict=True)\n    raw = Path(raw).absolute()\n    manifest_path = _volume_input(raw / \'manifest.json\', root)\n    marker = _volume_input(writer_closed_marker, root)\n    if sha256_file(marker) != expected_writer_closed_sha256:\n        raise ValueError(\'pinned closed writer receipt required\')\n    match = re.fullmatch(r\'(a[0-3])-raw-([A-Za-z0-9][A-Za-z0-9._-]{0,47})\', raw.parent.parent.name)\n    if match is None or raw.name != \'raw\' or raw.parent.name != \'payloads\':\n        raise ValueError(\'run-owned raw recovery path required\')\n    expected_operation = raw.parents[2] / (match[1] + \'-cpu-join-operation-\' + match[2]) / \'payloads\'\n    closed = json.loads(marker.read_text(encoding=\'utf-8\'))\n    if closed.get(\'kind\') == \'MODAL_CPU_WRITER_STOPPED_V1\':\n        config_path = _volume_input(expected_operation / \'config.json\', root)\n        config = json.loads(config_path.read_text(encoding=\'utf-8\'))\n        authorization = config[\'authorization\']\n        if (closed.get(\'status\') != \'APP_STOPPED\' or type(closed.get(\'tasks\')) is not int\n                or closed[\'tasks\'] != 0 or closed.get(\'stage\') != \'cpu-join\'\n                or not re.fullmatch(r\'ap-[A-Za-z0-9]+\', closed.get(\'app_id\', \'\'))\n                or closed.get(\'run_id\') != raw.parents[2].name\n                or closed.get(\'operation_tranche\') != expected_operation.parent.name\n                or closed.get(\'allocation_id\') != match[2]\n                or closed.get(\'request_sha256\') != authorization[\'request_sha256\']\n                or closed.get(\'operation_config_sha256\') != sha256_file(config_path)\n                or authorization[\'allocation_id\'] != match[2] or authorization[\'stage\'] != \'cpu-join\'\n                or config[\'stage\'] != \'cpu-join\' or config[\'request\'][\'run_id\'] != raw.parents[2].name\n                or config[\'request\'][\'variant\'] != match[1].upper()):\n            raise ValueError(\'external stop receipt must bind stopped app tasks0 and original writer authorization\')\n    elif (marker.name not in (\'FAILED.json\', \'COMPLETE.json\')\n            or marker.parent != expected_operation or closed.get(\'stage\') != \'cpu-join\'):\n        raise ValueError(\'closed marker must belong to this raw writer attempt\')\n    if sha256_file(manifest_path) != expected_manifest_sha256:\n        raise ValueError(\'raw manifest changed before recovery\')\n    manifest = json.loads(manifest_path.read_text(encoding=\'utf-8\'))\n    identity = manifest[\'identity\']\n    view.assert_integrity()\n    if (view.scope != \'calibration_predict\' or not view.full_role_coverage\n            or manifest[\'status\'] not in (\'incomplete\', \'complete\') or identity[\'phase\'] != \'junyi_response_calibration\'\n            or identity[\'checkpoint\'] != binding.checkpoint_sha256\n            or identity[\'inference_source\'] != binding.model_source_sha256\n            or identity[\'population\'] != population[\'identity\']\n            or manifest[\'prediction_rows\'] != population[\'targets\']\n            or len(manifest[\'completed_users\']) != len(view.users)\n            or set(manifest[\'completed_users\']) != set(view.users)):\n        raise ValueError(\'complete learner population and frozen raw identity required\')\n    if (manifest[\'status\'] == \'complete\'\n            and (manifest.get(\'expected_rows\') != population[\'targets\']\n                 or manifest.get(\'unique_users\') != len(view.users)\n                 or manifest.get(\'output_coverage\') != 1.0)):\n        raise ValueError(\'completed raw export lacks full audited coverage\')\n    for name in (\'canonical\', \'assignments\', \'canonical_manifest\', \'split_manifest\'):\n        _volume_input(getattr(paths, name), root)\n    _source_pins(paths, source_pins)\n    source = json.loads(paths.canonical_manifest.read_text(encoding=\'utf-8\'))\n    if (source.get(\'purpose\') != \'JUNYI_REFIT90_RESPONSE_CALIBRATION_SOURCE\'\n            or source.get(\'roles_identity\') != view.roles.identity\n            or source.get(\'phase_identity\') != view.identity\n            or source.get(\'encoding_identity\') != binding.vocabulary_sha256\n            or any(source.get(name) is not False for name in (\'train_events_included\',\n                \'selection_events_included\', \'outer_events_included\', \'holdout_events_included\'))):\n        raise ValueError(\'calibration10-only source required before recovery reader\')\n    shards = {}\n    for entry in manifest[\'shards\']:\n        name = entry[\'path\']\n        if Path(name).name != name or name in shards or not name.endswith(\'.parquet\'):\n            raise ValueError(\'unique raw shard basename required\')\n        path = _volume_input(raw / name, root)\n        shards[name] = dict(bytes=path.stat().st_size, sha256=sha256_file(path))\n    if sum(entry[\'rows\'] for entry in manifest[\'shards\']) != population[\'targets\']:\n        raise ValueError(\'raw shard target count mismatch\')\n    if {path.name for path in raw.glob(\'part-*.parquet\')} != set(shards):\n        raise ValueError(\'unlisted raw Parquet must be preserved for separate review\')\n    if any((raw / \'.pending\').glob(\'*.parquet.tmp\')):\n        raise ValueError(\'unlisted pending artifacts must be preserved for separate review\')\n    receipt_output.mkdir(parents=True, exist_ok=False)\n    prediction._write_json_atomic(receipt_output / \'BEFORE_MANIFEST.json\', manifest)\n    prediction._write_json_atomic(receipt_output / \'config.json\', dict(\n        method=\'original_export_predictions_empty_stream_with_expected_transaction\',\n        worker_source_sha256=sha256_file(Path(__file__)),\n        writer_closed_sha256=expected_writer_closed_sha256,\n        before_manifest_sha256=expected_manifest_sha256, source_pins=source_pins, shards=shards))\n    scratch = {}\n    for name in (\'.audit.tmp.duckdb\', \'.audit.tmp.duckdb.wal\'):\n        path = raw / name\n        if path.exists():\n            _volume_input(path, root)\n            scratch[name] = dict(bytes=path.stat().st_size, sha256=sha256_file(path))\n    if scratch:\n        saved = receipt_output / \'before-audit\'\n        saved.mkdir()\n        for name, pin in scratch.items():\n            (raw / name).rename(saved / name)\n            if sha256_file(saved / name) != pin[\'sha256\']:\n                raise ValueError(\'prior audit scratch changed during preservation\')\n    prediction._write_json_atomic(receipt_output / \'BEFORE_AUDIT_FILES.json\', scratch)\n    prepare = prediction._prepare_output\n\n    def prepare_pinned(output_dir, *, artifact_root=None):\n        if Path(output_dir).absolute() != raw or artifact_root is not None:\n            raise ValueError(\'recovery output hook only admits the pinned physical raw directory\')\n        if sha256_file(manifest_path) != expected_manifest_sha256:\n            raise ValueError(\'raw manifest changed before expected-row audit\')\n        return raw, manifest_path\n\n    started = time.monotonic()\n    prediction._prepare_output = prepare_pinned\n    try:\n        with transactional_expected_insert():\n            result = prediction.export_predictions(iter(()), raw,\n                expected_learners_factory=lambda: iter_junyi_phase_learners(paths, view), identity=identity)\n        for name, pin in shards.items():\n            if sha256_file(raw / name) != pin[\'sha256\']:\n                raise ValueError(\'raw shard bytes changed during recovery\')\n        _source_pins(paths, source_pins)\n        receipt = dict(status=\'CPU_RAW_AUDIT_RECOVERED_IDENTICAL_PARQUET\',\n            raw_manifest_sha256=sha256_file(manifest_path), prediction_rows=result[\'prediction_rows\'],\n            shards=len(shards), seconds=time.monotonic()-started)\n        prediction._write_json_atomic(receipt_output / \'COMPLETE.json\', receipt)\n        return receipt\n    except BaseException as exc:\n        prediction._write_json_atomic(receipt_output / \'FAILED.json\', dict(\n            error_type=type(exc).__name__, seconds=time.monotonic()-started))\n        raise\n    finally:\n        prediction._prepare_output = prepare\n\n\ndef recover_cpu_calibration(request, root, *, gpu_result, raw_source_pin, closed_marker_pin,\n        attempt_id, publish, volume_commit):\n    """Recovery CPU audit→archives→V1 từ raw đã có, dùng metadata và callback CPU."""\n    import zipfile\n    from dataclasses import asdict\n    from calibration_cloud_worker import (validate_request, load_inputs, pinned,\n        cpu_tranches, archive_numeric_snapshot)\n    from knowledge_tracing.akt.research_junyi_calibration import inference_source_identity\n    from knowledge_tracing.akt.research_junyi_numeric_predictions import NumericBinding\n    from knowledge_tracing.akt.research_junyi_inference import _identity\n    from knowledge_tracing.akt.cloud_calibration_fit import fit_volume_calibration\n    from knowledge_tracing.akt.filesystem import publish_new_file\n    from knowledge_tracing.response_calibration import PredictionBinding\n\n    validate_request(request, \'cpu-join\')\n    if not callable(publish) or not callable(volume_commit):\n        raise ValueError(\'CPU publication and durable Volume callbacks required\')\n    root = Path(root).resolve(strict=True)\n    assets_by_variant, paths, population = load_inputs(request, root)\n    assets = assets_by_variant[request[\'variant\']]\n    if inference_source_identity() != request[\'source_identity\']:\n        raise ValueError(\'frozen inference source changed before CPU recovery\')\n    raw_manifest = pinned(raw_source_pin, root)\n    if raw_manifest.name != \'manifest.json\':\n        raise ValueError(\'existing raw manifest pin required\')\n    raw = raw_manifest.parent\n    if not raw.parent.parent.name.startswith(request[\'variant\'].lower()+\'-raw-\'):\n        raise ValueError(\'recovery raw variant differs from request\')\n    close = pinned(closed_marker_pin, root)\n    gpu_receipt = pinned(dict(path=gpu_result[\'receipt_path\'], sha256=gpu_result[\'receipt_sha256\']), root)\n    numeric = json.loads(gpu_receipt.read_text(encoding=\'utf-8\'))\n    numeric_binding = NumericBinding(**numeric[\'binding\'])\n    numeric_binding.validate()\n    cache = json.loads(pinned(request[\'cache_receipt\'], root).read_text(encoding=\'utf-8\'))\n    encoding, view = assets[\'encoding\'], assets[\'calibration\']\n    binding = PredictionBinding(assets[\'index\'][\'model\'], assets[\'index\'][\'checkpoint_sha256\'],\n        encoding.identity, view.roles.identity,\n        _identity(dict(domain=\'junyi_exercises\', values=encoding.native.vocabulary.values)),\n        _identity(dict(domain=\'junyi_logged_response\', reference_bank=\'not_applicable\')),\n        request[\'source_identity\'], cache[\'source_data_sha256\'])\n    old_config_path = _volume_input(raw.parent / \'config.json\', root)\n    old_config = json.loads(old_config_path.read_text(encoding=\'utf-8\'))\n    expected_graph = None\n    if request[\'variant\'] != \'A0\':\n        from knowledge_tracing.akt.research_junyi_graph_cache import _digest, FORMAT\n        graph = cache[\'graphs\'][request[\'variant\']]\n        expected_graph = _digest((FORMAT, graph[\'sha256\'], json.dumps(graph[\'binding\'], sort_keys=True)))\n    if (old_config[\'binding\'] != asdict(binding) or old_config[\'numeric_binding\'] != asdict(numeric_binding)\n            or old_config[\'gpu_receipt_sha256\'] != gpu_result[\'receipt_sha256\']\n            or old_config[\'population_identity\'] != population[\'identity\']\n            or old_config[\'source_pins\'] != request[\'source_pins\']\n            or old_config[\'counts\'] != numeric[\'counts\']\n            or old_config[\'window_admission_sha256\'] != cache[\'windows\'][\'admission_sha256\']\n            or numeric[\'status\'] != \'COMPLETE\' or numeric[\'kind\'] != \'JUNYI_CACHED_NUMERIC_INFERENCE_V1\'\n            or numeric_binding.variant != request[\'variant\']\n            or numeric_binding.checkpoint_sha256 != binding.checkpoint_sha256\n            or numeric_binding.checkpoint_id != binding.checkpoint_sha256\n            or numeric_binding.source_sha256 != request[\'source_identity\']\n            or numeric_binding.encoding_sha256 != encoding.identity\n            or numeric_binding.phase_sha256 != view.identity\n            or numeric_binding.window_manifest_sha256 != cache[\'windows\'][\'manifest_sha256\']\n            or numeric[\'dataset_sha256\'] != population[\'identity\']\n            or numeric[\'source_data_sha256\'] != cache[\'source_data_sha256\']\n            or numeric[\'source_data_sha256\'] != sha256_file(paths.canonical)\n            or numeric[\'roles_sha256\'] != view.roles.identity\n            or numeric[\'window_admission_sha256\'] != cache[\'windows\'][\'admission_sha256\']\n            or numeric[\'graph_cache_identity\'] != expected_graph):\n        raise ValueError(\'original raw/GPU/cache/refit recovery bindings mismatch\')\n    cloud = CalibrationFitArtifactRoot(root)\n    tranches = cpu_tranches(request[\'variant\'], attempt_id)\n    audit_output = root / request[\'run_id\'] / (request[\'variant\'].lower()+\'-audit-\'+attempt_id) / \'payloads\'\n    audited = resume_completed_raw_audit(raw, paths, view, binding=binding, population=population,\n        source_pins=request[\'source_pins\'], expected_manifest_sha256=raw_source_pin[\'sha256\'],\n        writer_closed_marker=close, expected_writer_closed_sha256=closed_marker_pin[\'sha256\'],\n        artifact_root=cloud, receipt_output=audit_output)\n    volume_commit()\n    publications = []\n\n    def durable_publish(publication):\n        volume_commit()\n        receipt = publish(publication)\n        publications.append(receipt)\n        return receipt\n\n    durable_publish(archive_numeric_snapshot(request, numeric, gpu_receipt, root, attempt_id=attempt_id))\n    payloads = root / request[\'run_id\'] / tranches[\'raw\'] / \'payloads\'\n    cloud.validate_output(payloads)\n    payloads.mkdir(parents=True, exist_ok=False)\n    prediction._write_json_atomic(payloads/\'config.json\', dict(request=request,\n        recovery_source_raw=str(raw), before_raw_manifest_sha256=raw_source_pin[\'sha256\'],\n        old_raw_config_sha256=sha256_file(old_config_path), audit=audited,\n        worker_source_sha256=sha256_file(Path(__file__))))\n    manifest = json.loads(raw_manifest.read_text(encoding=\'utf-8\'))\n    names = [\'manifest.json\', *(entry[\'path\'] for entry in manifest[\'shards\'])]\n    inventory = {name: dict(bytes=(raw/name).stat().st_size, sha256=sha256_file(raw/name)) for name in names}\n    encoded = json.dumps(inventory, sort_keys=True)\n    estimated = sum(pin[\'bytes\'] for pin in inventory.values()) + len(encoded.encode()) + 512\n    estimated += sum(2*len((\'raw/\'+name).encode())+200 for name in names)\n    if estimated > 2*1024**3:\n        raise ValueError(\'recovered raw archive exceeds2GiB cap\')\n    partial = payloads/\'RAW_EXPORT.zip.partial\'\n    with zipfile.ZipFile(partial, \'x\', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:\n        for name in names:\n            archive.write(raw/name, \'raw/\'+name)\n            if sha256_file(raw/name) != inventory[name][\'sha256\']:\n                raise ValueError(\'recovered raw bytes changed while packing\')\n        archive.writestr(\'INVENTORY.json\', encoded)\n    archive_path = payloads/\'RAW_EXPORT.zip\'\n    if partial.stat().st_size > 2*1024**3:\n        raise ValueError(\'recovered raw archive actual byte cap exceeded\')\n    publish_new_file(partial, archive_path)\n    prediction._write_json_atomic(payloads/\'COMPLETE.json\', dict(status=\'RECOVERED_RAW_ARCHIVE_COMPLETE\',\n        raw_manifest_sha256=sha256_file(raw_manifest), archive_sha256=sha256_file(archive_path),\n        total_targets=manifest[\'prediction_rows\'], original_raw_path=str(raw)))\n    files = [\'config.json\', \'RAW_EXPORT.zip\', \'COMPLETE.json\']\n    total = sum((payloads/name).stat().st_size for name in files)\n    if total > 2*1024**3:\n        raise ValueError(\'recovery raw publish exceeds2GiB cap\')\n    durable_publish(dict(run_id=request[\'run_id\'], tranche_id=tranches[\'raw\'], files=files, max_total_bytes=total))\n    fitted = fit_volume_calibration(raw, paths, view, binding=binding, population=population,\n        expected_population_identity=population[\'identity\'], expected_manifest_sha256=sha256_file(raw_manifest),\n        shard_sha256={entry[\'path\']:sha256_file(raw/entry[\'path\']) for entry in manifest[\'shards\']},\n        source_pins=request[\'source_pins\'], max_targets=population[\'targets\'],\n        max_learner_events=request[\'cpu_budget\'][\'max_learner_events\'], artifact_root=cloud,\n        run_id=request[\'run_id\'], tranche_id=tranches[\'fit\'])\n    durable_publish(fitted[\'publish_request\'])\n    return dict(status=\'CPU_CALIBRATION_RECOVERY_PUBLISHED\', calibration_identity=fitted[\'calibration_identity\'],\n        audit=audited, publish_receipts=publications)\n'}
for name,text in FILES.items():
    (TOOLS/name).write_text(text,encoding="utf-8")
PAYLOAD={'requests': {'A2': {'kind': 'JUNYI_CLOUD_CALIBRATION_REQUEST_V1', 'workspace': 'honguyenminhsang2016', 'mode': 'full', 'run_id': 'junyi-calibration-real-20261117-01', 'variant': 'A2', 'exports': {'A0': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a0-refit-20261114-01/MODEL_INDEX.json', 'sha256': 'dcbca516c13be5c7c3d69609d389525a115fc53a03bfac9df4031cac3dd1caec'}, 'A1': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a1-refit-20261114-01/MODEL_INDEX.json', 'sha256': '39da8618c6de34c3419219ecc7ccf669cf648b2c5b052252afb60b9dec5834da'}, 'A2': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a2-refit-20261114-01/MODEL_INDEX.json', 'sha256': 'ac90435dc8c43ec65c64afe4dbce3f3d50d794841f3bb5cdd7d0f1dcc5c4c963'}, 'A3': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a3-refit-20261114-01/MODEL_INDEX.json', 'sha256': '9679ae6fb5e3cd1799ce16f7321a9b8ef9e7b7aa396ded7af0b54d9524298250'}}, 'data_paths': {'canonical': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/akt/junyi-cohorts/outer4-calibration-full-20261117-01/canonical.parquet', 'sha256': '7c8540d820578751f5675ec8c777aa87a6918b15a345bfa55f35ae6833b18692'}, 'assignments': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/akt/junyi-cohorts/outer4-calibration-full-20261117-01/assignments.parquet', 'sha256': '13cb082e53ecb24cb0d675d01ab80031bd6a68dbec98da636686b0f442023dd3'}, 'canonical_manifest': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/akt/junyi-cohorts/outer4-calibration-full-20261117-01/manifest.json', 'sha256': '5f01562926dbb4222b0133488a16a6babbadfc158d4dfc970a2d50fd7c7eb32a'}, 'split_manifest': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/bkt_baseline/splits/manifest.json', 'sha256': 'a69c0b440ef7d206133470f13696e853cb237c35c56c2b9015c175edaa89f94c'}}, 'population': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/research_tracking/runs/junyi-calibration-data-20261117-01/POPULATION.json', 'sha256': '8861748238569d32781f87900eab91826c1b54061157454af2b7e7381374fa59'}, 'source_pins': {'canonical_manifest': '5f01562926dbb4222b0133488a16a6babbadfc158d4dfc970a2d50fd7c7eb32a', 'split_manifest': 'a69c0b440ef7d206133470f13696e853cb237c35c56c2b9015c175edaa89f94c'}, 'cache_receipt': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/a0-cpu-cache-operation/payloads/result.json', 'sha256': '710d8cc4dff454b2922038a7dec5e74e4a1a1f91bde0fbc6cdc035f53ecbae4f'}, 'numeric_budget': {'max_seconds': 240.0, 'max_rss_bytes': 6442450944, 'max_vram_bytes': 8589934592, 'expected_records': 28653, 'expected_windows': 28653, 'expected_targets': 1621484, 'expected_oov_targets': 0, 'batch_size': 128}, 'cpu_budget': {'max_targets': 1621484, 'max_learner_events': 13491, 'max_users': 15847, 'max_seconds': 240.0, 'max_rss_bytes': 6442450944, 'min_available_ram_bytes': 268435456, 'prediction_batch_size': 128}, 'gpu_output': 'junyi-calibration-real-20261117-01/a2-numeric/payloads', 'full_cohort_approved': True, 'max_cost_usd': 0.75, 'source_identity': 'd22f6d2e3d6bb4a6c2d7f72da748e55c5c647ce8b41a7ab46b25c82290a4c96c'}, 'A3': {'kind': 'JUNYI_CLOUD_CALIBRATION_REQUEST_V1', 'workspace': 'honguyenminhsang2016', 'mode': 'full', 'run_id': 'junyi-calibration-real-20261117-01', 'variant': 'A3', 'exports': {'A0': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a0-refit-20261114-01/MODEL_INDEX.json', 'sha256': 'dcbca516c13be5c7c3d69609d389525a115fc53a03bfac9df4031cac3dd1caec'}, 'A1': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a1-refit-20261114-01/MODEL_INDEX.json', 'sha256': '39da8618c6de34c3419219ecc7ccf669cf648b2c5b052252afb60b9dec5834da'}, 'A2': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a2-refit-20261114-01/MODEL_INDEX.json', 'sha256': 'ac90435dc8c43ec65c64afe4dbce3f3d50d794841f3bb5cdd7d0f1dcc5c4c963'}, 'A3': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/experiments/akt/results/model-export-a3-refit-20261114-01/MODEL_INDEX.json', 'sha256': '9679ae6fb5e3cd1799ce16f7321a9b8ef9e7b7aa396ded7af0b54d9524298250'}}, 'data_paths': {'canonical': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/akt/junyi-cohorts/outer4-calibration-full-20261117-01/canonical.parquet', 'sha256': '7c8540d820578751f5675ec8c777aa87a6918b15a345bfa55f35ae6833b18692'}, 'assignments': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/akt/junyi-cohorts/outer4-calibration-full-20261117-01/assignments.parquet', 'sha256': '13cb082e53ecb24cb0d675d01ab80031bd6a68dbec98da636686b0f442023dd3'}, 'canonical_manifest': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/akt/junyi-cohorts/outer4-calibration-full-20261117-01/manifest.json', 'sha256': '5f01562926dbb4222b0133488a16a6babbadfc158d4dfc970a2d50fd7c7eb32a'}, 'split_manifest': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/artifacts/bkt_baseline/splits/manifest.json', 'sha256': 'a69c0b440ef7d206133470f13696e853cb237c35c56c2b9015c175edaa89f94c'}}, 'population': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/calibration-inputs-20261117-01/source/research_tracking/runs/junyi-calibration-data-20261117-01/POPULATION.json', 'sha256': '8861748238569d32781f87900eab91826c1b54061157454af2b7e7381374fa59'}, 'source_pins': {'canonical_manifest': '5f01562926dbb4222b0133488a16a6babbadfc158d4dfc970a2d50fd7c7eb32a', 'split_manifest': 'a69c0b440ef7d206133470f13696e853cb237c35c56c2b9015c175edaa89f94c'}, 'cache_receipt': {'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/a0-cpu-cache-operation/payloads/result.json', 'sha256': '710d8cc4dff454b2922038a7dec5e74e4a1a1f91bde0fbc6cdc035f53ecbae4f'}, 'numeric_budget': {'max_seconds': 240.0, 'max_rss_bytes': 6442450944, 'max_vram_bytes': 8589934592, 'expected_records': 28653, 'expected_windows': 28653, 'expected_targets': 1621484, 'expected_oov_targets': 0, 'batch_size': 128}, 'cpu_budget': {'max_targets': 1621484, 'max_learner_events': 13491, 'max_users': 15847, 'max_seconds': 240.0, 'max_rss_bytes': 6442450944, 'min_available_ram_bytes': 268435456, 'prediction_batch_size': 128}, 'gpu_output': 'junyi-calibration-real-20261117-01/a3-numeric/payloads', 'full_cohort_approved': True, 'max_cost_usd': 0.75, 'source_identity': 'd22f6d2e3d6bb4a6c2d7f72da748e55c5c647ce8b41a7ab46b25c82290a4c96c'}}, 'gpu': {'A2': {'counts': {'fitted_targets': 1621484, 'oov_targets': 0, 'records': 28653, 'targets': 1621484, 'windows': 28653}, 'host_phase_seconds': {'cache_receipt_read': 0.03772049599999994, 'config_write': 0.0007489250000034531, 'configure_cuda': 1.6766096160000004, 'graph_cache': 2.3353375709999993, 'imports': 0.31612062399999985, 'load_inputs': 2.1176158140000005, 'numeric_binding': 1.2760000000611171e-05, 'numeric_core': 29.688293326, 'request_validation': 5.4331000000296115e-05, 'source_checks': 0.007745777999999426, 'volume_commit': 1.2728149940000009, 'window_session': 0.7788837930000003}, 'receipt_path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/a2-numeric/payloads/RECEIPT.json', 'receipt_sha256': '26bd4b9ac3a9391d3a048afa06fc87439438842818e2f124629e4669204e96c5', 'status': 'GPU_NUMERIC_COMMITTED_RELEASE_GPU', 'volume_commit_seconds': 1.2728149940000009, 'worker_seconds': 38.231958028}, 'A3': {'counts': {'fitted_targets': 1621484, 'oov_targets': 0, 'records': 28653, 'targets': 1621484, 'windows': 28653}, 'host_phase_seconds': {'cache_receipt_read': 0.0707861059999999, 'config_write': 0.0015078440000024784, 'configure_cuda': 2.507077945999999, 'graph_cache': 2.390561388000001, 'imports': 0.3925671150000003, 'load_inputs': 2.335896494, 'numeric_binding': 0.00013334299999989696, 'numeric_core': 35.078602544, 'request_validation': 6.182099999962887e-05, 'source_checks': 0.019447071000000093, 'volume_commit': 0.798786088, 'window_session': 1.2660276279999998}, 'receipt_path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/a3-numeric/payloads/RECEIPT.json', 'receipt_sha256': '9244234cc5f8b221d240d4e61c484b917e91425261e17facb4a32d3e9146b8d9', 'status': 'GPU_NUMERIC_COMMITTED_RELEASE_GPU', 'volume_commit_seconds': 0.798786088, 'worker_seconds': 44.861455388}}, 'numeric': {'A2': {'status': 'REMOTE_STREAM_SHA256_VERIFIED', 'file_id': '1Y2egpzAZDM8A11vrpZnMIZNqqt_bZZXU', 'name': 'A2_NUMERIC_COMPLETE.zip', 'folder_id': '1tUKMQF8TDnFj8LY_KJjH6wC9DH5-bB-Z', 'bytes': 35773201, 'sha256': '7ab954fa8dfce17e895af0c5cf829b28da2cef38594c447054e46e8caea056a2', 'source_plan_sha256': 'c852deb7fc14bbaeb14aafc7e012a9f98f5f5960124ee7da0a34b303cd98d6b3', 'verified_at': '2026-11-17T08:25:52.225450+00:00', 'account': 'honguyenminhsang2005@gmail.com', 'source_members': 449, 'url': 'https://drive.google.com/file/d/1Y2egpzAZDM8A11vrpZnMIZNqqt_bZZXU/view'}, 'A3': {'status': 'REMOTE_STREAM_SHA256_VERIFIED', 'file_id': '1DqKeh8dHSYSOgSJWlvpI-4vmzF8ssLf8', 'name': 'A3_NUMERIC_COMPLETE.zip', 'folder_id': '1tUKMQF8TDnFj8LY_KJjH6wC9DH5-bB-Z', 'bytes': 35773285, 'sha256': '391a7ffb806a718945400afb9c4e06e72e0d8fa28b38dd7472a4fbe09a401dbe', 'source_plan_sha256': 'b46a2833ef661038e3dffb111ed1ea9493a8f89b8203c815b275b0c3f92402c0', 'verified_at': '2026-11-17T08:31:03.664889+00:00', 'account': 'honguyenminhsang2005@gmail.com', 'source_members': 449, 'url': 'https://drive.google.com/file/d/1DqKeh8dHSYSOgSJWlvpI-4vmzF8ssLf8/view'}}, 'cache': {'confidence_fitted_here': False, 'current_target_labels_in_gpu_windows': False, 'dataset_sha256': '008a6752151d1a868e15f4bbb890755be28ef311420b17f2d5058f3627eb06e6', 'encoding_identity': '4b8eb3e6bc5be9abd1419cf4e818b2e75dc86b82e59a0fe94e837c5f3299734f', 'graphs': {'A1': {'binding': {'condition_sha256': '68be103d6fc9a1ca4cf008c922fe70f6bef53a362e66d2eb5c236e84e3d116f4', 'dataset_sha256': '008a6752151d1a868e15f4bbb890755be28ef311420b17f2d5058f3627eb06e6', 'encoding_sha256': '4b8eb3e6bc5be9abd1419cf4e818b2e75dc86b82e59a0fe94e837c5f3299734f', 'format': 'junyi-a1-graph-context-cache-v1', 'phase_sha256': '93f5e449e2c47dbd8e0744b094ac1c1a0b8e5d82c9b9cda8b8b5f85d557308a7', 'sources': {'research_junyi_compact.py': 'af4862ffb62fe4e5d17360432b15b62a936c645caad6fb534ec2ee0d64e9526a', 'research_junyi_data.py': '525e080cfaf8bd1e5b5ca00b9b4b824e83203c732e08086f6918e76c64e0228d', 'research_junyi_graph.py': '73c40e0c757bf9772fba046e97384a574db997b2454871f3a336eda3f872c3b9', 'research_junyi_graph_cache.py': 'c0322a68d68d625d1f0eb874ab828f614a4d85121aef308596188338d6fc7b3d', 'types.py': '312e57e4fa507f12b449841969b3a9c2b35c592909178b3040102b254aa20304', 'windows.py': 'e8d00ea0f8b9d07bdc979575109c1f1fc22498d55410bff05a12559bebdb2068'}, 'window_manifest_sha256': '64197fe3b4314a36f430636ae82447ea8d9056ed78f681f1f8499e28a5e0703e'}, 'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/common-caches01/A1.sqlite', 'sha256': 'e97123f4f519f7e3c3aafacc357c95e3cdfa00caf83e3a2734f0b17c43c0fed1', 'targets': 1621484, 'windows': 28653}, 'A2': {'binding': {'condition_sha256': '39dca109dd0e413ddb38e3c382324ee0d0d02eca447f70fceb8739397c466c5c', 'dataset_sha256': '008a6752151d1a868e15f4bbb890755be28ef311420b17f2d5058f3627eb06e6', 'encoding_sha256': '4b8eb3e6bc5be9abd1419cf4e818b2e75dc86b82e59a0fe94e837c5f3299734f', 'format': 'junyi-confidence-graph-context-cache-v1', 'phase_sha256': '93f5e449e2c47dbd8e0744b094ac1c1a0b8e5d82c9b9cda8b8b5f85d557308a7', 'sources': {'research_junyi_compact.py': 'af4862ffb62fe4e5d17360432b15b62a936c645caad6fb534ec2ee0d64e9526a', 'research_junyi_data.py': '525e080cfaf8bd1e5b5ca00b9b4b824e83203c732e08086f6918e76c64e0228d', 'research_junyi_graph.py': '73c40e0c757bf9772fba046e97384a574db997b2454871f3a336eda3f872c3b9', 'research_junyi_graph_cache.py': 'c0322a68d68d625d1f0eb874ab828f614a4d85121aef308596188338d6fc7b3d', 'types.py': '312e57e4fa507f12b449841969b3a9c2b35c592909178b3040102b254aa20304', 'windows.py': 'e8d00ea0f8b9d07bdc979575109c1f1fc22498d55410bff05a12559bebdb2068'}, 'window_manifest_sha256': '64197fe3b4314a36f430636ae82447ea8d9056ed78f681f1f8499e28a5e0703e'}, 'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/common-caches01/A2.sqlite', 'sha256': '4a5633a3aeb4dcbed22e030c929e44afe7263598162987243e1f2b65026603ea', 'targets': 1621484, 'windows': 28653}, 'A3': {'binding': {'condition_sha256': '9e0fa34a9d513adcab2af0f8af98d8c6865d97b6ea441871fcbadd2e3b6f6e40', 'dataset_sha256': '008a6752151d1a868e15f4bbb890755be28ef311420b17f2d5058f3627eb06e6', 'encoding_sha256': '4b8eb3e6bc5be9abd1419cf4e818b2e75dc86b82e59a0fe94e837c5f3299734f', 'format': 'junyi-confidence-graph-context-cache-v1', 'phase_sha256': '93f5e449e2c47dbd8e0744b094ac1c1a0b8e5d82c9b9cda8b8b5f85d557308a7', 'sources': {'research_junyi_compact.py': 'af4862ffb62fe4e5d17360432b15b62a936c645caad6fb534ec2ee0d64e9526a', 'research_junyi_data.py': '525e080cfaf8bd1e5b5ca00b9b4b824e83203c732e08086f6918e76c64e0228d', 'research_junyi_graph.py': '73c40e0c757bf9772fba046e97384a574db997b2454871f3a336eda3f872c3b9', 'research_junyi_graph_cache.py': 'c0322a68d68d625d1f0eb874ab828f614a4d85121aef308596188338d6fc7b3d', 'types.py': '312e57e4fa507f12b449841969b3a9c2b35c592909178b3040102b254aa20304', 'windows.py': 'e8d00ea0f8b9d07bdc979575109c1f1fc22498d55410bff05a12559bebdb2068'}, 'window_manifest_sha256': '64197fe3b4314a36f430636ae82447ea8d9056ed78f681f1f8499e28a5e0703e'}, 'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/common-caches01/A3.sqlite', 'sha256': '0d08125a4c4e8a2efe18c16191f7f9cc260e258a79b57c3d9cb5ee6d12f84bfc', 'targets': 1621484, 'windows': 28653}}, 'phase_identity': '93f5e449e2c47dbd8e0744b094ac1c1a0b8e5d82c9b9cda8b8b5f85d557308a7', 'population': {'identity': '008a6752151d1a868e15f4bbb890755be28ef311420b17f2d5058f3627eb06e6', 'phase_identity': '93f5e449e2c47dbd8e0744b094ac1c1a0b8e5d82c9b9cda8b8b5f85d557308a7', 'population_sha256': 'c3eecb92277a265849111a9ab141b5fd198e530fbd986804b827ca16226bba16', 'prefix_keys_sha256': '3ea4f3af844fa3e7ad90dad91ace109a8e52cc62e75b5ecd68dbbfbc03509a92', 'roles_identity': '7c9a8102e399bdbc6569fc5a929faae525480c86fe0c60d2d805244a80b8e45a', 'schema': 'junyi-calibration-population-v1', 'source_pins': {'canonical_manifest': '5f01562926dbb4222b0133488a16a6babbadfc158d4dfc970a2d50fd7c7eb32a', 'split_manifest': 'a69c0b440ef7d206133470f13696e853cb237c35c56c2b9015c175edaa89f94c'}, 'targets': 1621484}, 'seconds': 81.071776397, 'source_data_sha256': '7c8540d820578751f5675ec8c777aa87a6918b15a345bfa55f35ae6833b18692', 'source_pins': {'canonical_manifest': '5f01562926dbb4222b0133488a16a6babbadfc158d4dfc970a2d50fd7c7eb32a', 'split_manifest': 'a69c0b440ef7d206133470f13696e853cb237c35c56c2b9015c175edaa89f94c'}, 'status': 'JUNYI_CALIBRATION_CPU_CACHES_COMPLETE', 'windows': {'admission_path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/common-caches01/WINDOW_ADMISSION.json', 'admission_sha256': '45651b51f0e327cb2e0a2e62b113b08699db0be6a547320b1214e2dd95c26a45', 'manifest_sha256': '64197fe3b4314a36f430636ae82447ea8d9056ed78f681f1f8499e28a5e0703e', 'path': '/__modal/volumes/vo-dv1975utRQbmagCjA0brSe/junyi-calibration-real-20261117-01/common-caches01/windows', 'summary': {'learners': 15847, 'records': 28653, 'target_rows': 1621484, 'unseen_targets': 0, 'window_tokens': 2902084, 'windows': 28653}}}, 'window_metadata': [{'path': 'WINDOW_MANIFEST.json', 'bytes': 17337, 'sha256': '64197fe3b4314a36f430636ae82447ea8d9056ed78f681f1f8499e28a5e0703e', 'volume_path': 'junyi-calibration-real-20261117-01/common-caches01/windows/manifest.json', 'file_id': '17_0N2a_g_UEJ5xf9M9-Bqr3qFEpKMU1z', 'folder_id': '1XNJZvRBw62nNh21Qev_Sxt6SsV5u5943'}, {'path': 'WINDOW_ADMISSION.json', 'bytes': 23893, 'sha256': '45651b51f0e327cb2e0a2e62b113b08699db0be6a547320b1214e2dd95c26a45', 'volume_path': 'junyi-calibration-real-20261117-01/common-caches01/WINDOW_ADMISSION.json', 'file_id': '1B365rNWV2PlopNgaYfsjNur_Gcs-CKz9', 'folder_id': '1XNJZvRBw62nNh21Qev_Sxt6SsV5u5943'}], 'input': {'file_id': '1GlPhrWOLwFNeUiv1unYaEATypWmV9_Vy', 'bytes': 134525060, 'sha256': 'a34cee8e961baf7056e8f16159d2b9a25afcca72f3c19b3a37bd0d711aab2d6a'}}
(TOOLS/"PLAN.json").write_text(json.dumps(PAYLOAD),encoding="utf-8")
sys.path.insert(0,str(TOOLS))
from free_cpu_calibration import download,VOLUME,sha
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
with AuthorizedSession(credentials) as session:
    account=session.get('https://www.googleapis.com/drive/v3/about',params={'fields':'user(emailAddress)'},timeout=60).json()
    assert account['user']['emailAddress']=='honguyenminhsang2005@gmail.com'
    archive=download(session,PAYLOAD['input'],Path('/content/kltn-free-cpu-input/INPUT.zip'))
from modal_calibration_input_stage import extract_verified_input
SOURCE=VOLUME/'calibration-inputs-20261117-01/source'
if not SOURCE.exists():
    extract_verified_input(archive,SOURCE)
else:
    import zipfile
    with zipfile.ZipFile(archive) as zipped:
        frozen=json.loads(zipped.read('CLOUD_INPUT_PLAN.json'))
    for pin in frozen['files']:
        target=SOURCE/pin['path']
        assert target.is_file() and target.stat().st_size==pin['bytes'] and sha(target)==pin['sha256']
upstream=SOURCE/"artifacts/akt/dependencies/upstream/akt.py"
upstream.parent.mkdir(parents=True,exist_ok=True)
upstream.write_bytes(base64.b64decode('aW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCmZyb20gdG9yY2gubm4uaW5pdCBpbXBvcnQgeGF2aWVyX3VuaWZvcm1fCmZyb20gdG9yY2gubm4uaW5pdCBpbXBvcnQgY29uc3RhbnRfCmZyb20gdG9yY2gubm4uaW5pdCBpbXBvcnQgeGF2aWVyX25vcm1hbF8KaW1wb3J0IG1hdGgKaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIGVudW0gaW1wb3J0IEludEVudW0KaW1wb3J0IG51bXB5IGFzIG5wCgpkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKCgpjbGFzcyBEaW0oSW50RW51bSk6CiAgICBiYXRjaCA9IDAKICAgIHNlcSA9IDEKICAgIGZlYXR1cmUgPSAyCgoKY2xhc3MgQUtUKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgbl9xdWVzdGlvbiwgbl9waWQsIGRfbW9kZWwsIG5fYmxvY2tzLAogICAgICAgICAgICAgICAgIGtxX3NhbWUsIGRyb3BvdXQsIG1vZGVsX3R5cGUsIGZpbmFsX2ZjX2RpbT01MTIsIG5faGVhZHM9OCwgZF9mZj0yMDQ4LCAgbDI9MWUtNSwgc2VwYXJhdGVfcWE9RmFsc2UpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgICIiIgogICAgICAgIElucHV0OgogICAgICAgICAgICBkX21vZGVsOiBkaW1lbnNpb24gb2YgYXR0ZW50aW9uIGJsb2NrCiAgICAgICAgICAgIGZpbmFsX2ZjX2RpbTogZGltZW5zaW9uIG9mIGZpbmFsIGZ1bGx5IGNvbm5lY3RlZCBuZXQgYmVmb3JlIHByZWRpY3Rpb24KICAgICAgICAgICAgbl9oZWFkczogbnVtYmVyIG9mIGhlYWRzIGluIG11bHRpLWhlYWRlZCBhdHRlbnRpb24KICAgICAgICAgICAgZF9mZiA6IGRpbWVuc2lvbiBmb3IgZnVsbHkgY29ubnRlY3RlZCBuZXQgaW5zaWRlIHRoZSBiYXNpYyBibG9jawogICAgICAgICIiIgogICAgICAgIHNlbGYubl9xdWVzdGlvbiA9IG5fcXVlc3Rpb24KICAgICAgICBzZWxmLmRyb3BvdXQgPSBkcm9wb3V0CiAgICAgICAgc2VsZi5rcV9zYW1lID0ga3Ffc2FtZQogICAgICAgIHNlbGYubl9waWQgPSBuX3BpZAogICAgICAgIHNlbGYubDIgPSBsMgogICAgICAgIHNlbGYubW9kZWxfdHlwZSA9IG1vZGVsX3R5cGUKICAgICAgICBzZWxmLnNlcGFyYXRlX3FhID0gc2VwYXJhdGVfcWEKICAgICAgICBlbWJlZF9sID0gZF9tb2RlbAogICAgICAgIGlmIHNlbGYubl9waWQgPiAwOgogICAgICAgICAgICBzZWxmLmRpZmZpY3VsdF9wYXJhbSA9IG5uLkVtYmVkZGluZyhzZWxmLm5fcGlkKzEsIDEpCiAgICAgICAgICAgIHNlbGYucV9lbWJlZF9kaWZmID0gbm4uRW1iZWRkaW5nKHNlbGYubl9xdWVzdGlvbisxLCBlbWJlZF9sKQogICAgICAgICAgICBzZWxmLnFhX2VtYmVkX2RpZmYgPSBubi5FbWJlZGRpbmcoMiAqIHNlbGYubl9xdWVzdGlvbiArIDEsIGVtYmVkX2wpCiAgICAgICAgIyBuX3F1ZXN0aW9uKzEgLGRfbW9kZWwKICAgICAgICBzZWxmLnFfZW1iZWQgPSBubi5FbWJlZGRpbmcoc2VsZi5uX3F1ZXN0aW9uKzEsIGVtYmVkX2wpCiAgICAgICAgaWYgc2VsZi5zZXBhcmF0ZV9xYToKICAgICAgICAgICAgc2VsZi5xYV9lbWJlZCA9IG5uLkVtYmVkZGluZygyKnNlbGYubl9xdWVzdGlvbisxLCBlbWJlZF9sKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYucWFfZW1iZWQgPSBubi5FbWJlZGRpbmcoMiwgZW1iZWRfbCkKICAgICAgICAjIEFyY2hpdGVjdHVyZSBPYmplY3QuIEl0IGNvbnRhaW5zIHN0YWNrIG9mIGF0dGVudGlvbiBibG9jawogICAgICAgIHNlbGYubW9kZWwgPSBBcmNoaXRlY3R1cmUobl9xdWVzdGlvbj1uX3F1ZXN0aW9uLCBuX2Jsb2Nrcz1uX2Jsb2Nrcywgbl9oZWFkcz1uX2hlYWRzLCBkcm9wb3V0PWRyb3BvdXQsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRfbW9kZWw9ZF9tb2RlbCwgZF9mZWF0dXJlPWRfbW9kZWwgLyBuX2hlYWRzLCBkX2ZmPWRfZmYsICBrcV9zYW1lPXNlbGYua3Ffc2FtZSwgbW9kZWxfdHlwZT1zZWxmLm1vZGVsX3R5cGUpCgogICAgICAgIHNlbGYub3V0ID0gbm4uU2VxdWVudGlhbCgKICAgICAgICAgICAgbm4uTGluZWFyKGRfbW9kZWwgKyBlbWJlZF9sLAogICAgICAgICAgICAgICAgICAgICAgZmluYWxfZmNfZGltKSwgbm4uUmVMVSgpLCBubi5Ecm9wb3V0KHNlbGYuZHJvcG91dCksCiAgICAgICAgICAgIG5uLkxpbmVhcihmaW5hbF9mY19kaW0sIDI1NiksIG5uLlJlTFUoCiAgICAgICAgICAgICksIG5uLkRyb3BvdXQoc2VsZi5kcm9wb3V0KSwKICAgICAgICAgICAgbm4uTGluZWFyKDI1NiwgMSkKICAgICAgICApCiAgICAgICAgc2VsZi5yZXNldCgpCgogICAgZGVmIHJlc2V0KHNlbGYpOgogICAgICAgIGZvciBwIGluIHNlbGYucGFyYW1ldGVycygpOgogICAgICAgICAgICBpZiBwLnNpemUoMCkgPT0gc2VsZi5uX3BpZCsxIGFuZCBzZWxmLm5fcGlkID4gMDoKICAgICAgICAgICAgICAgIHRvcmNoLm5uLmluaXQuY29uc3RhbnRfKHAsIDAuKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHFfZGF0YSwgcWFfZGF0YSwgdGFyZ2V0LCBwaWRfZGF0YT1Ob25lKToKICAgICAgICAjIEJhdGNoIEZpcnN0CiAgICAgICAgcV9lbWJlZF9kYXRhID0gc2VsZi5xX2VtYmVkKHFfZGF0YSkgICMgQlMsIHNlcWxlbiwgIGRfbW9kZWwjIGNfY3QKICAgICAgICBpZiBzZWxmLnNlcGFyYXRlX3FhOgogICAgICAgICAgICAjIEJTLCBzZXFsZW4sIGRfbW9kZWwgI2ZfKGN0LHJ0KQogICAgICAgICAgICBxYV9lbWJlZF9kYXRhID0gc2VsZi5xYV9lbWJlZChxYV9kYXRhKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHFhX2RhdGEgPSAocWFfZGF0YS1xX2RhdGEpLy9zZWxmLm5fcXVlc3Rpb24gICMgcnQKICAgICAgICAgICAgIyBCUywgc2VxbGVuLCBkX21vZGVsICMgY19jdCsgZ19ydCA9ZV8oY3QscnQpCiAgICAgICAgICAgIHFhX2VtYmVkX2RhdGEgPSBzZWxmLnFhX2VtYmVkKHFhX2RhdGEpK3FfZW1iZWRfZGF0YQoKICAgICAgICBpZiBzZWxmLm5fcGlkID4gMDoKICAgICAgICAgICAgcV9lbWJlZF9kaWZmX2RhdGEgPSBzZWxmLnFfZW1iZWRfZGlmZihxX2RhdGEpICAjIGRfY3QKICAgICAgICAgICAgcGlkX2VtYmVkX2RhdGEgPSBzZWxmLmRpZmZpY3VsdF9wYXJhbShwaWRfZGF0YSkgICMgdXEKICAgICAgICAgICAgcV9lbWJlZF9kYXRhID0gcV9lbWJlZF9kYXRhICsgcGlkX2VtYmVkX2RhdGEgKiBcCiAgICAgICAgICAgICAgICBxX2VtYmVkX2RpZmZfZGF0YSAgIyB1cSAqZF9jdCArIGNfY3QKICAgICAgICAgICAgcWFfZW1iZWRfZGlmZl9kYXRhID0gc2VsZi5xYV9lbWJlZF9kaWZmKAogICAgICAgICAgICAgICAgcWFfZGF0YSkgICMgZl8oY3QscnQpIG9yICNoX3J0CiAgICAgICAgICAgIGlmIHNlbGYuc2VwYXJhdGVfcWE6CiAgICAgICAgICAgICAgICBxYV9lbWJlZF9kYXRhID0gcWFfZW1iZWRfZGF0YSArIHBpZF9lbWJlZF9kYXRhICogXAogICAgICAgICAgICAgICAgICAgIHFhX2VtYmVkX2RpZmZfZGF0YSAgIyB1cSogZl8oY3QscnQpICsgZV8oY3QscnQpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBxYV9lbWJlZF9kYXRhID0gcWFfZW1iZWRfZGF0YSArIHBpZF9lbWJlZF9kYXRhICogXAogICAgICAgICAgICAgICAgICAgIChxYV9lbWJlZF9kaWZmX2RhdGErcV9lbWJlZF9kaWZmX2RhdGEpICAjICsgdXEgKihoX3J0K2RfY3QpCiAgICAgICAgICAgIGNfcmVnX2xvc3MgPSAocGlkX2VtYmVkX2RhdGEgKiogMi4pLnN1bSgpICogc2VsZi5sMgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNfcmVnX2xvc3MgPSAwLgoKICAgICAgICAjIEJTLnNlcWxlbixkX21vZGVsCiAgICAgICAgIyBQYXNzIHRvIHRoZSBkZWNvZGVyCiAgICAgICAgIyBvdXRwdXQgc2hhcGUgQlMsc2VxbGVuLGRfbW9kZWwgb3IgZF9tb2RlbC8vMgogICAgICAgIGRfb3V0cHV0ID0gc2VsZi5tb2RlbChxX2VtYmVkX2RhdGEsIHFhX2VtYmVkX2RhdGEpICAjIDIxMXg1MTIKCiAgICAgICAgY29uY2F0X3EgPSB0b3JjaC5jYXQoW2Rfb3V0cHV0LCBxX2VtYmVkX2RhdGFdLCBkaW09LTEpCiAgICAgICAgb3V0cHV0ID0gc2VsZi5vdXQoY29uY2F0X3EpCiAgICAgICAgbGFiZWxzID0gdGFyZ2V0LnJlc2hhcGUoLTEpCiAgICAgICAgbSA9IG5uLlNpZ21vaWQoKQogICAgICAgIHByZWRzID0gKG91dHB1dC5yZXNoYXBlKC0xKSkgICMgbG9naXQKICAgICAgICBtYXNrID0gbGFiZWxzID4gLTAuOQogICAgICAgIG1hc2tlZF9sYWJlbHMgPSBsYWJlbHNbbWFza10uZmxvYXQoKQogICAgICAgIG1hc2tlZF9wcmVkcyA9IHByZWRzW21hc2tdCiAgICAgICAgbG9zcyA9IG5uLkJDRVdpdGhMb2dpdHNMb3NzKHJlZHVjdGlvbj0nbm9uZScpCiAgICAgICAgb3V0cHV0ID0gbG9zcyhtYXNrZWRfcHJlZHMsIG1hc2tlZF9sYWJlbHMpCiAgICAgICAgcmV0dXJuIG91dHB1dC5zdW0oKStjX3JlZ19sb3NzLCBtKHByZWRzKSwgbWFzay5zdW0oKQoKCmNsYXNzIEFyY2hpdGVjdHVyZShubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG5fcXVlc3Rpb24sICBuX2Jsb2NrcywgZF9tb2RlbCwgZF9mZWF0dXJlLAogICAgICAgICAgICAgICAgIGRfZmYsIG5faGVhZHMsIGRyb3BvdXQsIGtxX3NhbWUsIG1vZGVsX3R5cGUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgICIiIgogICAgICAgICAgICBuX2Jsb2NrIDogbnVtYmVyIG9mIHN0YWNrZWQgYmxvY2tzIGluIHRoZSBhdHRlbnRpb24KICAgICAgICAgICAgZF9tb2RlbCA6IGRpbWVuc2lvbiBvZiBhdHRlbnRpb24gaW5wdXQvb3V0cHV0CiAgICAgICAgICAgIGRfZmVhdHVyZSA6IGRpbWVuc2lvbiBvZiBpbnB1dCBpbiBlYWNoIG9mIHRoZSBtdWx0aS1oZWFkIGF0dGVudGlvbiBwYXJ0LgogICAgICAgICAgICBuX2hlYWQgOiBudW1iZXIgb2YgaGVhZHMuIG5faGVhZHMqZF9mZWF0dXJlID0gZF9tb2RlbAogICAgICAgICIiIgogICAgICAgIHNlbGYuZF9tb2RlbCA9IGRfbW9kZWwKICAgICAgICBzZWxmLm1vZGVsX3R5cGUgPSBtb2RlbF90eXBlCgogICAgICAgIGlmIG1vZGVsX3R5cGUgaW4geydha3QnfToKICAgICAgICAgICAgc2VsZi5ibG9ja3NfMSA9IG5uLk1vZHVsZUxpc3QoWwogICAgICAgICAgICAgICAgVHJhbnNmb3JtZXJMYXllcihkX21vZGVsPWRfbW9kZWwsIGRfZmVhdHVyZT1kX21vZGVsIC8vIG5faGVhZHMsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRfZmY9ZF9mZiwgZHJvcG91dD1kcm9wb3V0LCBuX2hlYWRzPW5faGVhZHMsIGtxX3NhbWU9a3Ffc2FtZSkKICAgICAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKG5fYmxvY2tzKQogICAgICAgICAgICBdKQogICAgICAgICAgICBzZWxmLmJsb2Nrc18yID0gbm4uTW9kdWxlTGlzdChbCiAgICAgICAgICAgICAgICBUcmFuc2Zvcm1lckxheWVyKGRfbW9kZWw9ZF9tb2RlbCwgZF9mZWF0dXJlPWRfbW9kZWwgLy8gbl9oZWFkcywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZF9mZj1kX2ZmLCBkcm9wb3V0PWRyb3BvdXQsIG5faGVhZHM9bl9oZWFkcywga3Ffc2FtZT1rcV9zYW1lKQogICAgICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9ibG9ja3MqMikKICAgICAgICAgICAgXSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBxX2VtYmVkX2RhdGEsIHFhX2VtYmVkX2RhdGEpOgogICAgICAgICMgdGFyZ2V0IHNoYXBlICBicywgc2VxbGVuCiAgICAgICAgc2VxbGVuLCBiYXRjaF9zaXplID0gcV9lbWJlZF9kYXRhLnNpemUoMSksIHFfZW1iZWRfZGF0YS5zaXplKDApCgogICAgICAgIHFhX3Bvc19lbWJlZCA9IHFhX2VtYmVkX2RhdGEKICAgICAgICBxX3Bvc19lbWJlZCA9IHFfZW1iZWRfZGF0YQoKICAgICAgICB5ID0gcWFfcG9zX2VtYmVkCiAgICAgICAgc2VxbGVuLCBiYXRjaF9zaXplID0geS5zaXplKDEpLCB5LnNpemUoMCkKICAgICAgICB4ID0gcV9wb3NfZW1iZWQKCiAgICAgICAgIyBlbmNvZGVyCiAgICAgICAgZm9yIGJsb2NrIGluIHNlbGYuYmxvY2tzXzE6ICAjIGVuY29kZSBxYXMKICAgICAgICAgICAgeSA9IGJsb2NrKG1hc2s9MSwgcXVlcnk9eSwga2V5PXksIHZhbHVlcz15KQogICAgICAgIGZsYWdfZmlyc3QgPSBUcnVlCiAgICAgICAgZm9yIGJsb2NrIGluIHNlbGYuYmxvY2tzXzI6CiAgICAgICAgICAgIGlmIGZsYWdfZmlyc3Q6ICAjIHBlZWsgY3VycmVudCBxdWVzdGlvbgogICAgICAgICAgICAgICAgeCA9IGJsb2NrKG1hc2s9MSwgcXVlcnk9eCwga2V5PXgsCiAgICAgICAgICAgICAgICAgICAgICAgICAgdmFsdWVzPXgsIGFwcGx5X3Bvcz1GYWxzZSkKICAgICAgICAgICAgICAgIGZsYWdfZmlyc3QgPSBGYWxzZQogICAgICAgICAgICBlbHNlOiAgIyBkb250IHBlZWsgY3VycmVudCByZXNwb25zZQogICAgICAgICAgICAgICAgeCA9IGJsb2NrKG1hc2s9MCwgcXVlcnk9eCwga2V5PXgsIHZhbHVlcz15LCBhcHBseV9wb3M9VHJ1ZSkKICAgICAgICAgICAgICAgIGZsYWdfZmlyc3QgPSBUcnVlCiAgICAgICAgcmV0dXJuIHgKCgpjbGFzcyBUcmFuc2Zvcm1lckxheWVyKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZF9tb2RlbCwgZF9mZWF0dXJlLAogICAgICAgICAgICAgICAgIGRfZmYsIG5faGVhZHMsIGRyb3BvdXQsICBrcV9zYW1lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICAiIiIKICAgICAgICAgICAgVGhpcyBpcyBhIEJhc2ljIEJsb2NrIG9mIFRyYW5zZm9ybWVyIHBhcGVyLiBJdCBjb250YWludHMgb25lIE11bHRpLWhlYWQgYXR0ZW50aW9uIG9iamVjdC4gRm9sbG93ZWQgYnkgbGF5ZXIgbm9ybSBhbmQgcG9zdGlvbiB3aXNlIGZlZWRmb3J3YXJkIG5ldCBhbmQgZHJvcG91dCBsYXllci4KICAgICAgICAiIiIKICAgICAgICBrcV9zYW1lID0ga3Ffc2FtZSA9PSAxCiAgICAgICAgIyBNdWx0aS1IZWFkIEF0dGVudGlvbiBCbG9jawogICAgICAgIHNlbGYubWFza2VkX2F0dG5faGVhZCA9IE11bHRpSGVhZEF0dGVudGlvbigKICAgICAgICAgICAgZF9tb2RlbCwgZF9mZWF0dXJlLCBuX2hlYWRzLCBkcm9wb3V0LCBrcV9zYW1lPWtxX3NhbWUpCgogICAgICAgICMgVHdvIGxheWVyIG5vcm0gbGF5ZXIgYW5kIHR3byBkcm9wdXQgbGF5ZXIKICAgICAgICBzZWxmLmxheWVyX25vcm0xID0gbm4uTGF5ZXJOb3JtKGRfbW9kZWwpCiAgICAgICAgc2VsZi5kcm9wb3V0MSA9IG5uLkRyb3BvdXQoZHJvcG91dCkKCiAgICAgICAgc2VsZi5saW5lYXIxID0gbm4uTGluZWFyKGRfbW9kZWwsIGRfZmYpCiAgICAgICAgc2VsZi5hY3RpdmF0aW9uID0gbm4uUmVMVSgpCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYubGluZWFyMiA9IG5uLkxpbmVhcihkX2ZmLCBkX21vZGVsKQoKICAgICAgICBzZWxmLmxheWVyX25vcm0yID0gbm4uTGF5ZXJOb3JtKGRfbW9kZWwpCiAgICAgICAgc2VsZi5kcm9wb3V0MiA9IG5uLkRyb3BvdXQoZHJvcG91dCkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCBtYXNrLCBxdWVyeSwga2V5LCB2YWx1ZXMsIGFwcGx5X3Bvcz1UcnVlKToKICAgICAgICAiIiIKICAgICAgICBJbnB1dDoKICAgICAgICAgICAgYmxvY2sgOiBvYmplY3Qgb2YgdHlwZSBCYXNpY0Jsb2NrKG5uLk1vZHVsZSkuIEl0IGNvbnRhaW5zIG1hc2tlZF9hdHRuX2hlYWQgb2JqZWN0cyB3aGljaCBpcyBvZiB0eXBlIE11bHRpSGVhZEF0dGVudGlvbihubi5Nb2R1bGUpLgogICAgICAgICAgICBtYXNrIDogMCBtZWFucywgaXQgY2FuIHBlZWsgb25seSBwYXN0IHZhbHVlcy4gMSBtZWFucywgYmxvY2sgY2FuIHBlZWsgb25seSBjdXJyZW50IGFuZCBwYXMgdmFsdWVzCiAgICAgICAgICAgIHF1ZXJ5IDogUXVlcnkuIEluIHRyYW5zZm9ybWVyIHBhcGVyIGl0IGlzIHRoZSBpbnB1dCBmb3IgYm90aCBlbmNvZGVyIGFuZCBkZWNvZGVyCiAgICAgICAgICAgIGtleSA6IEtleXMuIEluIHRyYW5zZm9ybWVyIHBhcGVyIGl0IGlzIHRoZSBpbnB1dCBmb3IgYm90aCBlbmNvZGVyIGFuZCBkZWNvZGVyCiAgICAgICAgICAgIFZhbHVlcy4gSW4gdHJhbnNmb3JtZXIgcGFwZXIgaXQgaXMgdGhlIGlucHV0IGZvciBlbmNvZGVyIGFuZCAgZW5jb2RlZCBvdXRwdXQgZm9yIGRlY29kZXIgKGluIG1hc2tlZCBhdHRlbnRpb24gcGFydCkKCiAgICAgICAgT3V0cHV0OgogICAgICAgICAgICBxdWVyeTogSW5wdXQgZ2V0cyBjaGFuZ2VkIG92ZXIgdGhlIGxheWVyIGFuZCByZXR1cm5lZC4KCiAgICAgICAgIiIiCgogICAgICAgIHNlcWxlbiwgYmF0Y2hfc2l6ZSA9IHF1ZXJ5LnNpemUoMSksIHF1ZXJ5LnNpemUoMCkKICAgICAgICBub3BlZWtfbWFzayA9IG5wLnRyaXUoCiAgICAgICAgICAgIG5wLm9uZXMoKDEsIDEsIHNlcWxlbiwgc2VxbGVuKSksIGs9bWFzaykuYXN0eXBlKCd1aW50OCcpCiAgICAgICAgc3JjX21hc2sgPSAodG9yY2guZnJvbV9udW1weShub3BlZWtfbWFzaykgPT0gMCkudG8oZGV2aWNlKQogICAgICAgIGlmIG1hc2sgPT0gMDogICMgSWYgMCwgemVyby1wYWRkaW5nIGlzIG5lZWRlZC4KICAgICAgICAgICAgIyBDYWxscyBibG9jay5tYXNrZWRfYXR0bl9oZWFkLmZvcndhcmQoKSBtZXRob2QKICAgICAgICAgICAgcXVlcnkyID0gc2VsZi5tYXNrZWRfYXR0bl9oZWFkKAogICAgICAgICAgICAgICAgcXVlcnksIGtleSwgdmFsdWVzLCBtYXNrPXNyY19tYXNrLCB6ZXJvX3BhZD1UcnVlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgICMgQ2FsbHMgYmxvY2subWFza2VkX2F0dG5faGVhZC5mb3J3YXJkKCkgbWV0aG9kCiAgICAgICAgICAgIHF1ZXJ5MiA9IHNlbGYubWFza2VkX2F0dG5faGVhZCgKICAgICAgICAgICAgICAgIHF1ZXJ5LCBrZXksIHZhbHVlcywgbWFzaz1zcmNfbWFzaywgemVyb19wYWQ9RmFsc2UpCgogICAgICAgIHF1ZXJ5ID0gcXVlcnkgKyBzZWxmLmRyb3BvdXQxKChxdWVyeTIpKQogICAgICAgIHF1ZXJ5ID0gc2VsZi5sYXllcl9ub3JtMShxdWVyeSkKICAgICAgICBpZiBhcHBseV9wb3M6CiAgICAgICAgICAgIHF1ZXJ5MiA9IHNlbGYubGluZWFyMihzZWxmLmRyb3BvdXQoCiAgICAgICAgICAgICAgICBzZWxmLmFjdGl2YXRpb24oc2VsZi5saW5lYXIxKHF1ZXJ5KSkpKQogICAgICAgICAgICBxdWVyeSA9IHF1ZXJ5ICsgc2VsZi5kcm9wb3V0MigocXVlcnkyKSkKICAgICAgICAgICAgcXVlcnkgPSBzZWxmLmxheWVyX25vcm0yKHF1ZXJ5KQogICAgICAgIHJldHVybiBxdWVyeQoKCmNsYXNzIE11bHRpSGVhZEF0dGVudGlvbihubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWwsIGRfZmVhdHVyZSwgbl9oZWFkcywgZHJvcG91dCwga3Ffc2FtZSwgYmlhcz1UcnVlKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICAiIiIKICAgICAgICBJdCBoYXMgcHJvamVjdGlvbiBsYXllciBmb3IgZ2V0dGluZyBrZXlzLCBxdWVyaWVzIGFuZCB2YWx1ZXMuIEZvbGxvd2VkIGJ5IGF0dGVudGlvbiBhbmQgYSBjb25uZWN0ZWQgbGF5ZXIuCiAgICAgICAgIiIiCiAgICAgICAgc2VsZi5kX21vZGVsID0gZF9tb2RlbAogICAgICAgIHNlbGYuZF9rID0gZF9mZWF0dXJlCiAgICAgICAgc2VsZi5oID0gbl9oZWFkcwogICAgICAgIHNlbGYua3Ffc2FtZSA9IGtxX3NhbWUKCiAgICAgICAgc2VsZi52X2xpbmVhciA9IG5uLkxpbmVhcihkX21vZGVsLCBkX21vZGVsLCBiaWFzPWJpYXMpCiAgICAgICAgc2VsZi5rX2xpbmVhciA9IG5uLkxpbmVhcihkX21vZGVsLCBkX21vZGVsLCBiaWFzPWJpYXMpCiAgICAgICAgaWYga3Ffc2FtZSBpcyBGYWxzZToKICAgICAgICAgICAgc2VsZi5xX2xpbmVhciA9IG5uLkxpbmVhcihkX21vZGVsLCBkX21vZGVsLCBiaWFzPWJpYXMpCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYucHJval9iaWFzID0gYmlhcwogICAgICAgIHNlbGYub3V0X3Byb2ogPSBubi5MaW5lYXIoZF9tb2RlbCwgZF9tb2RlbCwgYmlhcz1iaWFzKQogICAgICAgIHNlbGYuZ2FtbWFzID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKG5faGVhZHMsIDEsIDEpKQogICAgICAgIHRvcmNoLm5uLmluaXQueGF2aWVyX3VuaWZvcm1fKHNlbGYuZ2FtbWFzKQoKICAgICAgICBzZWxmLl9yZXNldF9wYXJhbWV0ZXJzKCkKCiAgICBkZWYgX3Jlc2V0X3BhcmFtZXRlcnMoc2VsZik6CiAgICAgICAgeGF2aWVyX3VuaWZvcm1fKHNlbGYua19saW5lYXIud2VpZ2h0KQogICAgICAgIHhhdmllcl91bmlmb3JtXyhzZWxmLnZfbGluZWFyLndlaWdodCkKICAgICAgICBpZiBzZWxmLmtxX3NhbWUgaXMgRmFsc2U6CiAgICAgICAgICAgIHhhdmllcl91bmlmb3JtXyhzZWxmLnFfbGluZWFyLndlaWdodCkKCiAgICAgICAgaWYgc2VsZi5wcm9qX2JpYXM6CiAgICAgICAgICAgIGNvbnN0YW50XyhzZWxmLmtfbGluZWFyLmJpYXMsIDAuKQogICAgICAgICAgICBjb25zdGFudF8oc2VsZi52X2xpbmVhci5iaWFzLCAwLikKICAgICAgICAgICAgaWYgc2VsZi5rcV9zYW1lIGlzIEZhbHNlOgogICAgICAgICAgICAgICAgY29uc3RhbnRfKHNlbGYucV9saW5lYXIuYmlhcywgMC4pCiAgICAgICAgICAgIGNvbnN0YW50XyhzZWxmLm91dF9wcm9qLmJpYXMsIDAuKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHEsIGssIHYsIG1hc2ssIHplcm9fcGFkKToKCiAgICAgICAgYnMgPSBxLnNpemUoMCkKCiAgICAgICAgIyBwZXJmb3JtIGxpbmVhciBvcGVyYXRpb24gYW5kIHNwbGl0IGludG8gaCBoZWFkcwoKICAgICAgICBrID0gc2VsZi5rX2xpbmVhcihrKS52aWV3KGJzLCAtMSwgc2VsZi5oLCBzZWxmLmRfaykKICAgICAgICBpZiBzZWxmLmtxX3NhbWUgaXMgRmFsc2U6CiAgICAgICAgICAgIHEgPSBzZWxmLnFfbGluZWFyKHEpLnZpZXcoYnMsIC0xLCBzZWxmLmgsIHNlbGYuZF9rKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHEgPSBzZWxmLmtfbGluZWFyKHEpLnZpZXcoYnMsIC0xLCBzZWxmLmgsIHNlbGYuZF9rKQogICAgICAgIHYgPSBzZWxmLnZfbGluZWFyKHYpLnZpZXcoYnMsIC0xLCBzZWxmLmgsIHNlbGYuZF9rKQoKICAgICAgICAjIHRyYW5zcG9zZSB0byBnZXQgZGltZW5zaW9ucyBicyAqIGggKiBzbCAqIGRfbW9kZWwKCiAgICAgICAgayA9IGsudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgcSA9IHEudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgdiA9IHYudHJhbnNwb3NlKDEsIDIpCiAgICAgICAgIyBjYWxjdWxhdGUgYXR0ZW50aW9uIHVzaW5nIGZ1bmN0aW9uIHdlIHdpbGwgZGVmaW5lIG5leHQKICAgICAgICBnYW1tYXMgPSBzZWxmLmdhbW1hcwogICAgICAgIHNjb3JlcyA9IGF0dGVudGlvbihxLCBrLCB2LCBzZWxmLmRfaywKICAgICAgICAgICAgICAgICAgICAgICAgICAgbWFzaywgc2VsZi5kcm9wb3V0LCB6ZXJvX3BhZCwgZ2FtbWFzKQoKICAgICAgICAjIGNvbmNhdGVuYXRlIGhlYWRzIGFuZCBwdXQgdGhyb3VnaCBmaW5hbCBsaW5lYXIgbGF5ZXIKICAgICAgICBjb25jYXQgPSBzY29yZXMudHJhbnNwb3NlKDEsIDIpLmNvbnRpZ3VvdXMoKVwKICAgICAgICAgICAgLnZpZXcoYnMsIC0xLCBzZWxmLmRfbW9kZWwpCgogICAgICAgIG91dHB1dCA9IHNlbGYub3V0X3Byb2ooY29uY2F0KQoKICAgICAgICByZXR1cm4gb3V0cHV0CgoKZGVmIGF0dGVudGlvbihxLCBrLCB2LCBkX2ssIG1hc2ssIGRyb3BvdXQsIHplcm9fcGFkLCBnYW1tYT1Ob25lKToKICAgICIiIgogICAgVGhpcyBpcyBjYWxsZWQgYnkgTXVsdGktaGVhZCBhdGVudGlvbiBvYmplY3QgdG8gZmluZCB0aGUgdmFsdWVzLgogICAgIiIiCiAgICBzY29yZXMgPSB0b3JjaC5tYXRtdWwocSwgay50cmFuc3Bvc2UoLTIsIC0xKSkgLyBcCiAgICAgICAgbWF0aC5zcXJ0KGRfaykgICMgQlMsIDgsIHNlcWxlbiwgc2VxbGVuCiAgICBicywgaGVhZCwgc2VxbGVuID0gc2NvcmVzLnNpemUoMCksIHNjb3Jlcy5zaXplKDEpLCBzY29yZXMuc2l6ZSgyKQoKICAgIHgxID0gdG9yY2guYXJhbmdlKHNlcWxlbikuZXhwYW5kKHNlcWxlbiwgLTEpLnRvKGRldmljZSkKICAgIHgyID0geDEudHJhbnNwb3NlKDAsIDEpLmNvbnRpZ3VvdXMoKQoKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIHNjb3Jlc18gPSBzY29yZXMubWFza2VkX2ZpbGwobWFzayA9PSAwLCAtMWUzMikKICAgICAgICBzY29yZXNfID0gRi5zb2Z0bWF4KHNjb3Jlc18sIGRpbT0tMSkgICMgQlMsOCxzZXFsZW4sc2VxbGVuCiAgICAgICAgc2NvcmVzXyA9IHNjb3Jlc18gKiBtYXNrLmZsb2F0KCkudG8oZGV2aWNlKQogICAgICAgIGRpc3RjdW1fc2NvcmVzID0gdG9yY2guY3Vtc3VtKHNjb3Jlc18sIGRpbT0tMSkgICMgYnMsIDgsIHNsLCBzbAogICAgICAgIGRpc3R0b3RhbF9zY29yZXMgPSB0b3JjaC5zdW0oCiAgICAgICAgICAgIHNjb3Jlc18sIGRpbT0tMSwga2VlcGRpbT1UcnVlKSAgIyBicywgOCwgc2wsIDEKICAgICAgICBwb3NpdGlvbl9lZmZlY3QgPSB0b3JjaC5hYnMoCiAgICAgICAgICAgIHgxLXgyKVtOb25lLCBOb25lLCA6LCA6XS50eXBlKHRvcmNoLkZsb2F0VGVuc29yKS50byhkZXZpY2UpICAjIDEsIDEsIHNlcWxlbiwgc2VxbGVuCiAgICAgICAgIyBicywgOCwgc2wsIHNsIHBvc2l0aXZlIGRpc3RhbmNlCiAgICAgICAgZGlzdF9zY29yZXMgPSB0b3JjaC5jbGFtcCgKICAgICAgICAgICAgKGRpc3R0b3RhbF9zY29yZXMtZGlzdGN1bV9zY29yZXMpKnBvc2l0aW9uX2VmZmVjdCwgbWluPTAuKQogICAgICAgIGRpc3Rfc2NvcmVzID0gZGlzdF9zY29yZXMuc3FydCgpLmRldGFjaCgpCiAgICBtID0gbm4uU29mdHBsdXMoKQogICAgZ2FtbWEgPSAtMS4gKiBtKGdhbW1hKS51bnNxdWVlemUoMCkgICMgMSw4LDEsMQogICAgIyBOb3cgYWZ0ZXIgZG8gZXhwKGdhbW1hKmRpc3RhbmNlKSBhbmQgdGhlbiBjbGFtcCB0byAxZS01IHRvIDFlNQogICAgdG90YWxfZWZmZWN0ID0gdG9yY2guY2xhbXAodG9yY2guY2xhbXAoCiAgICAgICAgKGRpc3Rfc2NvcmVzKmdhbW1hKS5leHAoKSwgbWluPTFlLTUpLCBtYXg9MWU1KQogICAgc2NvcmVzID0gc2NvcmVzICogdG90YWxfZWZmZWN0CgogICAgc2NvcmVzLm1hc2tlZF9maWxsXyhtYXNrID09IDAsIC0xZTMyKQogICAgc2NvcmVzID0gRi5zb2Z0bWF4KHNjb3JlcywgZGltPS0xKSAgIyBCUyw4LHNlcWxlbixzZXFsZW4KICAgIGlmIHplcm9fcGFkOgogICAgICAgIHBhZF96ZXJvID0gdG9yY2guemVyb3MoYnMsIGhlYWQsIDEsIHNlcWxlbikudG8oZGV2aWNlKQogICAgICAgIHNjb3JlcyA9IHRvcmNoLmNhdChbcGFkX3plcm8sIHNjb3Jlc1s6LCA6LCAxOiwgOl1dLCBkaW09MikKICAgIHNjb3JlcyA9IGRyb3BvdXQoc2NvcmVzKQogICAgb3V0cHV0ID0gdG9yY2gubWF0bXVsKHNjb3JlcywgdikKICAgIHJldHVybiBvdXRwdXQKCgpjbGFzcyBMZWFybmFibGVQb3NpdGlvbmFsRW1iZWRkaW5nKG5uLk1vZHVsZSk6CiAgICBkZWYgX19pbml0X18oc2VsZiwgZF9tb2RlbCwgbWF4X2xlbj01MTIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgICMgQ29tcHV0ZSB0aGUgcG9zaXRpb25hbCBlbmNvZGluZ3Mgb25jZSBpbiBsb2cgc3BhY2UuCiAgICAgICAgcGUgPSAwLjEgKiB0b3JjaC5yYW5kbihtYXhfbGVuLCBkX21vZGVsKQogICAgICAgIHBlID0gcGUudW5zcXVlZXplKDApCiAgICAgICAgc2VsZi53ZWlnaHQgPSBubi5QYXJhbWV0ZXIocGUsIHJlcXVpcmVzX2dyYWQ9VHJ1ZSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4gc2VsZi53ZWlnaHRbOiwgOnguc2l6ZShEaW0uc2VxKSwgOl0gICMgKCAxLHNlcSwgIEZlYXR1cmUpCgoKY2xhc3MgQ29zaW5lUG9zaXRpb25hbEVtYmVkZGluZyhubi5Nb2R1bGUpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRfbW9kZWwsIG1heF9sZW49NTEyKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICAjIENvbXB1dGUgdGhlIHBvc2l0aW9uYWwgZW5jb2RpbmdzIG9uY2UgaW4gbG9nIHNwYWNlLgogICAgICAgIHBlID0gMC4xICogdG9yY2gucmFuZG4obWF4X2xlbiwgZF9tb2RlbCkKICAgICAgICBwb3NpdGlvbiA9IHRvcmNoLmFyYW5nZSgwLCBtYXhfbGVuKS51bnNxdWVlemUoMSkuZmxvYXQoKQogICAgICAgIGRpdl90ZXJtID0gdG9yY2guZXhwKHRvcmNoLmFyYW5nZSgwLCBkX21vZGVsLCAyKS5mbG9hdCgpICoKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAtKG1hdGgubG9nKDEwMDAwLjApIC8gZF9tb2RlbCkpCiAgICAgICAgcGVbOiwgMDo6Ml0gPSB0b3JjaC5zaW4ocG9zaXRpb24gKiBkaXZfdGVybSkKICAgICAgICBwZVs6LCAxOjoyXSA9IHRvcmNoLmNvcyhwb3NpdGlvbiAqIGRpdl90ZXJtKQogICAgICAgIHBlID0gcGUudW5zcXVlZXplKDApCiAgICAgICAgc2VsZi53ZWlnaHQgPSBubi5QYXJhbWV0ZXIocGUsIHJlcXVpcmVzX2dyYWQ9RmFsc2UpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgcmV0dXJuIHNlbGYud2VpZ2h0WzosIDp4LnNpemUoRGltLnNlcSksIDpdICAjICggMSxzZXEsICBGZWF0dXJlKQo='))
venv_path=SOURCE/'.venv'
if not (venv_path/'bin/python').exists():
    venv.EnvBuilder(with_pip=False).create(venv_path)
PYTHON=str(venv_path/'bin/python')
import shutil
UV=shutil.which('uv')
assert UV, 'PREINSTALLED_UV_REQUIRED'
env=dict(os.environ,UV_CACHE_DIR=str(SOURCE/'artifacts/akt/uv-cache'))
subprocess.run([UV,'pip','install','--python',PYTHON,'numpy==2.5.4','pandas==3.0.6','duckdb==1.5.6','psutil==7.2.2','requests==2.34.2','google-auth==2.62.1','scikit-learn==1.9.1','scipy==1.18.1','pyarrow==26.0.1'],env=env,check=True)
assert not Path('/artifacts').exists() or Path('/artifacts').resolve()==VOLUME
if not Path('/artifacts').exists():
    Path('/artifacts').symlink_to(VOLUME,target_is_directory=True)
print('CPU_SOURCE_AND_VENV_PREPARED')


In [ ]:
(TOOLS/"RUN_CPU.py").write_text('import json,sys,os\nfrom pathlib import Path\nTOOLS=Path(\'/content/kltn-free-cpu-tools\')\nsys.path.insert(0,str(TOOLS))\nfrom free_cpu_calibration import VOLUME,RUN,download,extract_numeric,run_variant,sha,need\nSOURCE=VOLUME/\'calibration-inputs-20261117-01/source\'\nsys.path.insert(0,str(SOURCE/\'src\'))\nimport psutil,google.auth\nfrom google.auth.transport.requests import AuthorizedSession\nfrom knowledge_tracing.akt.research_junyi_calibration import inference_source_identity\nplan=json.loads((TOOLS/\'PLAN.json\').read_text())\nneed(inference_source_identity()==plan[\'requests\'][\'A2\'][\'source_identity\'],\'Frozen source mismatch\')\nneed(\'torch\' not in sys.modules,\'CPU launcher unexpectedly imported Torch\')\nneed(psutil.virtual_memory().available>=4*1024**3,\'CPU free RAM below estimated 4GiB working budget\')\nstate_root=Path(\'/content/drive/MyDrive/KLTN_AKT_PRIVATE/calibration-free-cpu-20261117-01\')\nstate_root.mkdir(parents=True,exist_ok=True)\ncredentials,_=google.auth.default(scopes=[\'https://www.googleapis.com/auth/drive\'])\nwith AuthorizedSession(credentials) as session:\n    account=session.get(\'https://www.googleapis.com/drive/v3/about\',params={\'fields\':\'user(emailAddress)\'},timeout=60).json()\n    need(account[\'user\'][\'emailAddress\']==\'honguyenminhsang2005@gmail.com\',\'Drive account mismatch\')\n    private=session.get(\'https://www.googleapis.com/drive/v3/files\',params={\'q\':"\'root\' in parents and name=\'KLTN_AKT_PRIVATE\' and trashed=false",\'fields\':\'files(id,shared,mimeType)\'},timeout=60)\n    need(private.status_code==200,\'Private state folder lookup failed\')\n    private_folders=private.json().get(\'files\',[])\n    need(len(private_folders)==1 and not private_folders[0].get(\'shared\',True),\'Private state folder must be unshared MyDrive folder\')\n    windows=plan[\'cache\'][\'windows\']\n    for pin in plan[\'window_metadata\']:\n        target=Path(windows[\'path\'])/\'manifest.json\' if pin[\'path\']==\'WINDOW_MANIFEST.json\' else Path(windows[\'admission_path\'])\n        download(session,pin,target)\n    for variant in (\'A2\',\'A3\'):\n        archive=download(session,plan[\'numeric\'][variant],Path(\'/content/kltn-free-cpu-input\')/(variant+\'_NUMERIC.zip\'))\n        extract_numeric(archive,plan[\'numeric\'][variant],VOLUME,variant)\n        print(variant,\'CPU_JOIN_AUDIT_FIT_START\',flush=True)\n        from drive_resumable_upload import single_writer\n        with single_writer(state_root/(variant+\'.lock\')):\n            result=run_variant(plan[\'requests\'][variant],plan[\'gpu\'][variant],plan[\'cache\'],session,state_root)\n        print(variant,\'DRIVE_CALIBRATION_COMPLETE\',result[\'fit_verified\'],flush=True)\n        temperature=next(p for p in result[\'fit_publication\'][\'files\'] if p[\'path\']==\'fit/temperature.json\')\n        response=session.get(\'https://www.googleapis.com/drive/v3/files/\'+temperature[\'file_id\'],params={\'alt\':\'media\'},timeout=60)\n        import hashlib\n        need(response.status_code==200 and len(response.content)==temperature[\'bytes\'] and hashlib.sha256(response.content).hexdigest()==temperature[\'sha256\'],\'Temperature readback mismatch\')\n        summary=response.json()\n        print(json.dumps({k:summary[k] for k in (\'total_targets\',\'supported_targets\',\'temperature\',\'raw_log_loss\',\'calibrated_log_loss\',\'partition_sha256\')},sort_keys=True),flush=True)\n',encoding="utf-8")
subprocess.run([PYTHON,"-B",str(TOOLS/"RUN_CPU.py")],check=True)


## Checkpoint vận hành

Mỗi variant chỉ được ghi hoàn tất khi đủ raw đã audit, temperature/manifest/config/resource/COMPLETE và SHA Drive được đọc lại. Đây là calibration fit, kết quả độc lập outer-fold vẫn còn bước khác. Khi lỗi, giữ state và scratch, sửa kết nối rồi chạy lại ô cuối. Không bấm tạo runtime mới nếu chỉ mất mạng. Nếu runtime bị cấp lại, chạy lại từ đầu để tải input đã pin và tiếp tục từ raw đã publish.

Runtime này không giữ GPU cần trả. Khi hoàn tất, kiểm receipt trên Drive trước khi ngắt runtime CPU. Notebook không xóa bất kỳ model/checkpoint nguồn nào.